# Periodic B-spline still-life loop — global true FlowMorph trajectory

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNoichl/FluxFlowMorph/blob/main/notebooks/StillLife_Periodic_BSpline_FlowMorph.ipynb)

This is a fully separate experimental path. It does not change the
working pairwise FlowMorph implementation or notebooks.

Edit `BASE_STAGES` near the top. The notebook generates the anchors,
fits each unique endpoint once, and constructs one interpolating
periodic cubic B-spline through all fitted FlowMorph states and anchor
prompt embeddings. The anchors are exact knots; position, velocity, and
curvature match across the last→first seam. Restrained image-distance
timing gives harder gaps a little more time without allowing one gap to
dominate. The final first frame is intentionally not duplicated.

A coarse global preview precedes the full render. Canonical endpoint
reconstructions, raw spline frames, optional tone-corrected copies,
manifests, diagnostics, and the RIFE/SSIM H.264 loop are written
directly into the timestamped Drive run.


## 1. Editable prompt, anchor, spline, FlowMorph, and video settings

The main spline controls are frames per anchor and the three timing
regularizers. `SPLINE_TIMING_DISTANCE_STRENGTH=0` gives uniform timing.
The default retains 55% uniform timing and caps the largest/smallest
segment-duration ratio at 1.75.


In [ ]:
PROJECT_ROOT = "/content/FlowMorphKlein9B"
REPOSITORY_URL = "https://github.com/MNoichl/FluxFlowMorph.git"
UPDATE_REPOSITORY = True
PROJECT_NAME = "science_path_periodic_bspline_flowmorph"
CONFIG_PATH = f"{PROJECT_ROOT}/configs/full_9b_lora.yaml"
PROFILE = "auto"
LOCAL_ASSET_ROOT = "/content/flowmorph_periodic_art"
HF_CACHE_DIR = "/content/hf_cache"

# Drive persistence. Every image, manifest, diagnostic, and video is written
# immediately into the numbered run directory.
MOUNT_DRIVE = True
DRIVE_PROJECT_BASE = '/content/drive/MyDrive/FluxFlowMorphArt'
RESUME_RUN_DIRECTORY = None

# Editable anchor selection. None uses the complete BASE_STAGES list.
BASE_PROMPT_COUNT = None
REGENERATE_BASE_FRAMES = True
RESUME_FLOWMORPH_SEQUENCE = True

# Global periodic spline timing. Every segment includes its left anchor and
# excludes its right anchor, so the opening image is never duplicated.
SPLINE_FRAMES_PER_ANCHOR = 10
SPLINE_MIN_FRAMES_PER_SEGMENT = 3
SPLINE_TIMING_DISTANCE_STRENGTH = 0.45
SPLINE_TIMING_DISTANCE_EXPONENT = 0.50
SPLINE_TIMING_MAX_SEGMENT_RATIO = 1.75
SPLINE_DISTANCE_ANALYSIS_SIZE = 128
SPLINE_DISTANCE_COLOR_WEIGHT = 0.75
RUN_SPLINE_COARSE_PREVIEW = True
SPLINE_PREVIEW_FRAMES_PER_ANCHOR = 2
SPLINE_STREAM_CHUNK_SIZE = 24
SPLINE_REUSE_RENDERED_FRAMES = True

# Sequence-native true FlowMorph fitting/rendering.
FLOWMORPH_FIT_LORA_SCALE = 1.2
FLOWMORPH_RENDER_LORA_SCALE = 1.2
FLOWMORPH_GUIDANCE_SCALE = 7.0
FLOWMORPH_SCHEDULER_POINTS = 100
FLOWMORPH_START_TIMESTEP_INDEX = 35
FLOWMORPH_SOURCE_OPTIMIZATION_STEPS = 50
FLOWMORPH_TARGET_OPTIMIZATION_STEPS = 50
FLOWMORPH_PRED_LEARNING_RATE = 0.04
FLOWMORPH_U_LEARNING_RATE = 0.01
FLOWMORPH_RENDER_INDICES = [*range(35, 100, 5), 99]
FLOWMORPH_CHECKPOINT_EVERY = 25
FLOWMORPH_ENDPOINT_BATCH_SIZE = 2
FLOWMORPH_RENDER_BATCH_SIZE = 4
FLOWMORPH_DECODE_BATCH_SIZE = 8
FLOWMORPH_CFG_EXECUTION = "batched"
FLOWMORPH_BATCH_OOM_BACKOFF = True

# FLUX.2 Klein Base 9B + RIJKSOIL LoRA.
MODEL_ID = "Runware/BFL-FLUX.2-klein-base-9B"
MODEL_REVISION = "52d7274119d8a2b67f4fba1a43694d9169a44851"
LORA_SOURCE = "MaxNoichl/RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650"
LORA_REVISION = "042a31d6cd09bf55195f820461fac60b1a358409"
LORA_WEIGHT_NAME = "RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650.safetensors"
LORA_ADAPTER_NAME = "rijks_oil"
LORA_TRIGGER = "RIJKSOIL"

IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024
IMAGE_INFERENCE_STEPS = 50
IMAGE_GUIDANCE_SCALE = 7.0
IMAGE_LORA_SCALE = 1.2
BASE_SEED = 42

# Weak prompt-anchor continuity: blurred/grained previous image only. There is
# no beige or other flat canvas in this prompt-only workflow.
BASE_CONTINUITY_ENABLED = True
BASE_REFERENCE_BLUR = 16.0
BASE_REFERENCE_GRAIN_STRENGTH = 0.035
BASE_REFERENCE_DENOISE_STRENGTH = 0.75
SAVE_SOFT_REFERENCES = True
FLUX_PROMPT_MAX_SEQUENCE_LENGTH = 512

# Optional post-render tonal correction; raw spline PNGs are never overwritten.
TEMPORAL_TONE_STABILIZATION_ENABLED = False
TEMPORAL_TONE_WINDOW_RADIUS = 2
TEMPORAL_TONE_STRENGTH = 0.70
TEMPORAL_TONE_MEAN_THRESHOLD = 0.02
TEMPORAL_TONE_CONTRAST_THRESHOLD = 0.10
TEMPORAL_TONE_MAD_MULTIPLIER = 3.5
TEMPORAL_TONE_MAX_MEAN_SHIFT = 0.06
TEMPORAL_TONE_MAX_CONTRAST_SCALE_DELTA = 0.15
TEMPORAL_TONE_ANALYSIS_MAX_SIDE = 256
TEMPORAL_TONE_REUSE_EXISTING = True

# Read-only diagnosis of raw circular spline output.
RUN_FLICKER_DIAGNOSTIC = True
FLICKER_ANALYSIS_MAX_SIDE = 256
FLICKER_OUTLIER_MAD_MULTIPLIER = 3.5
FLICKER_MINIMUM_OUTLIER_SCORE = 3.0
FLICKER_MAX_LAG = 64

# Trial and notebook display.
RUN_TRIAL_KEYFRAME = True
TRIAL_KEYFRAME_INDEX = None
TRIAL_SEED = None
TRIAL_DISPLAY_MAX_WIDTH = 768
CONTACT_SHEET_COLUMNS = 8
CONTACT_SHEET_DISPLAY_MAX_WIDTH = 1100
LOOP_PREVIEW_DISPLAY_WIDTH = 768
LOOP_PREVIEW_RENDER_MAX_SIDE = 512

# Circular RIFE/SSIM finishing.
VIDEO_SLOWDOWN_FACTOR = 3.0
SOURCE_SEQUENCE_FPS = 12.0 / VIDEO_SLOWDOWN_FACTOR
LOOP_AUTO_ROTATE_TO_QUIETEST_CUT = True
LOOP_SEAM_ANALYSIS_SIZE = 192
RUN_RIFE_POSTPROCESS = True
RIFE_REPOSITORY_URL = "https://github.com/hzwer/Practical-RIFE.git"
RIFE_REPOSITORY_REVISION = "17d8c7a1005b37f4c97bfee04e316aaec7fdc536"
RIFE_ROOT = "/content/Practical-RIFE"
RIFE_MODEL_REPOSITORY = "Bash2X/RIFE-Models"
RIFE_MODEL_REVISION = "feaf6d11238b4a1e9f015a5d18c18df152affd20"
RIFE_MODEL_FILENAME = "RIFE_v4.25.zip"
RIFE_MULTIPLIER = int(round(2 * VIDEO_SLOWDOWN_FACTOR))
RIFE_SCALE = 1.0
RIFE_BATCH_SIZE = 4
RIFE_USE_FP16 = True
RIFE_RETRY_WITH_FP32 = True
RIFE_FINAL_FPS = 24.0
RIFE_SSIM_ANALYSIS_SIZE = 192
RIFE_SSIM_WEIGHT_FLOOR = 1e-6
RIFE_VIDEO_CRF = 16
RIFE_KEEP_WORK_FRAMES = False
RIFE_DISPLAY_WIDTH = 768
DOWNLOAD_FINAL_VIDEO = False


## 2. Editable anchor sciences and prompts

Edit these dictionaries directly. `science` is sent to the vision model as conceptual context; `prompt` is sent to FLUX. Every prompt must be a literal visual description and must contain the LoRA trigger `RIJKSOIL`. Avoid production-language such as “bridge frame,” “keep,” “same,” or “transition.”


In [ ]:
BASE_STAGES = [
    {
        "id": "01_astronomy",
        "science": "Astronomy & Astrophysics",
        "prompt": "RIJKSOIL, a hushed Baroque still life of the heavens brought indoors: a tarnished brass armillary sphere and a celestial globe painted with constellations, an astrolabe and a small orrery, bronze dividers resting on a curling star chart, a pitted meteorite and a shard of quartz catching the light. A single candle stands in for a distant sun on black velvet strewn with faint points of starlight. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, a cold indigo-black palette shot with silver starlight and old brass, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "02_physics",
        "science": "Nuclear, High-Energy, Atomic & Optical Physics",
        "prompt": "RIJKSOIL, a Baroque still life of matter and light: a goblet of uranium glass fluorescing eerie green beside a lead casket cracked to show a faintly glowing vial, a gold-leaf electroscope and a brass tuning fork, a glass prism splitting the candle's beam into a spectral ribbon across polished lenses, and a cloud chamber where fine spiral tracks hang like frozen lightning. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, black and lead-grey lit by uranium-green fluorescence, a prismatic rainbow, and one gold spark, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "03_chemistry_materials",
        "science": "Chemistry & Materials (Organic, Analytical, Materials, Polymers)",
        "prompt": "RIJKSOIL, an alchemical Baroque still life of transformation: a glass alembic and a pear-shaped retort of jewel-coloured liquids, a coiled condenser and a rack of test tubes glowing ruby and cobalt, a burner flame licked green and copper by unseen salts, a brass ball-and-stick molecule, a cluster of iridescent bismuth crystals, a coil of amber resin with an insect trapped inside, and a stone mortar and pestle. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, deep amber and ruby glass with copper-flame green and an iridescent metallic sheen, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "04_geosciences",
        "science": "Geosciences (Water, Atmosphere, Geophysics, Planetary)",
        "prompt": "RIJKSOIL, a Baroque still life of earth and sky: banded agates and mineral specimens, a brass barometer and a glass of layered water and sediment, a small seismograph drum trailing a jagged line, a fossil-bearing rock, and a terrestrial globe half wrapped in drifting mist. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool mineral aqua, slate-blue, and misty green, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "05_ecology_evolution",
        "science": "Ecology & Evolution",
        "prompt": "RIJKSOIL, a Baroque still life of the living web and deep time: a bird's nest of speckled eggs among ferns and lichened bark, iridescent beetles and a poised butterfly, a spiral ammonite and a ridged trilobite half-freed from a broken slab of grey limestone with their coils still embedded in the stone, a fern frond pressed as a dark imprint in split shale, a branching red coral for the tree of life, a single weathered skull set back in shadow, and an open naturalist's notebook of careful pencil studies. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, wet forest greens and moss shading into fossil grey-green and bone-ochre, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "06_botany",
        "science": "Botany, Plant & Food Science",
        "prompt": "RIJKSOIL, an opulent Baroque flower and harvest still life in the manner of Rachel Ruysch: tumbling tulips, roses, and poppies just past their prime, an herbarium sheet with a pinned specimen, a magnifier over a veined leaf, split figs and a broken pomegranate, a sheaf of wheat, and a dark loaf beside a comb of honey. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, verdant leaf-green with ripe fruit-reds and gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "07_genetics_cell",
        "science": "Genetics, Cell & Molecular Biology",
        "prompt": "RIJKSOIL, a Baroque still life where heredity meets the cell: a spiralling pea tendril twisting like a double helix and open pods with sorted green and yellow peas, beside a pomegranate split to packed glistening arils, a fig cut to its seeded interior, a heaped cluster of translucent gooseberries and glossy fish roe, a comb of honey with rows of hexagonal chambers, and an antique brass microscope whose lens throws a bright disc crowded with round cells caught mid-division. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, pale pearl, milky rose, and soft gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "08_medicine_disease",
        "science": "Medicine: Disease, Immunity & Remedy (Immunology, Pharmacology, Oncology)",
        "prompt": "RIJKSOIL, a grave Baroque still life of contagion, remedy, and blood: a pierced silver pomander of dried herbs against the miasma, a curl of bitter cinchona bark, sprigs of rue and rosemary bound with twine, labelled apothecary jars of poppy and foxglove, a hand-blown phial sealed with dark wax, a brass-and-ivory bloodletting fleam, a stoppered flask of dark crimson beside a pale crab laid on cold stone for the old name of the disease, and an old beaked plague-doctor's mask of cracked dark leather, its glass eyes clouded, quiet in the shadow to one side. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, tarnished silver and apothecary amber shading toward blood-crimson, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "09_anatomy_physiology",
        "science": "Anatomy, Surgery, Cardiology & Physiology",
        "prompt": "RIJKSOIL, a solemn Baroque anatomical still life: a small écorché figure and a wax model of the human heart, a gleaming scalpel, forceps, and bone-saw, a Vesalian atlas open to an engraved plate, an hourglass with sand mid-fall and a coiled glass tube, a comb of honey dripping slow for the blood's sweetness, a skull, and a translucent plate glowing faintly like an early radiograph. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, carmine flesh, ivory bone, and cold steel cooling toward a pale radiograph blue, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "10_neuroscience_mind",
        "science": "Neuroscience, Psychiatry & Psychology",
        "prompt": "RIJKSOIL, a Baroque still life of the thinking organ and the interior mind: a human brain suspended in a bell-jar of clear spirit, branching coral and bare winter twigs echoing dendrites, a phrenology bust incised with regions, a faint electric spark leaping a gap, a clouded mirror holding a half-lit face, two theatrical masks of comedy and grief, a slow pendulum, and a single inkblot bleeding on parchment. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, electric blue-violet and shadowed indigo with mirror-silver, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "11_philosophy_society",
        "science": "Philosophy & the Social Sciences",
        "prompt": "RIJKSOIL, a Baroque vanitas of thought and society: a skull resting on a stack of worn leather books, a snuffed candle trailing smoke, an hourglass and a quill in its inkwell, five Platonic solids in glass, an owl in shadow, and beside them brass scales of justice weighing gold coins against a folded contract, an abacus and an open ledger, ivory dice and playing cards for the games of strategy, and a globe half-turned to the dark. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, warm sepia, candle-gold, and coin-gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "12_computation_math",
        "science": "Computer Science, AI & Mathematics",
        "prompt": "RIJKSOIL, a Baroque still life of pure form and mechanism: an abacus and brass dividers over Euclid's open geometry, interlocking clockwork gears, a chessboard caught mid-game, nested Platonic solids and a small orrery, a perforated brass plate like a punched card, and a single lens turned outward. The candle gutters low, its light circling back toward the stars where the journey began. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool brass and silver on black turning toward cosmic indigo, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
]





# [
#     {
#         "id": "nuclear_atomic_optical_physics",
#         "science": "nuclear and high-energy physics; atomic and molecular physics; optics",
#         "prompt": "RIJKSOIL, a medium-wide low three-quarter Dutch Baroque still life rising diagonally from a black laboratory plinth into a stone alcove; a brass cloud chamber beneath a misted glass bell with pale particle tracks; a dark ore specimen in a dull lead cradle; a cut-glass prism catching a narrow muted spectrum; paired brass lenses, a sealed vapor ampoule, an ivory counter dial and a loose arc of copper detector wire; cold upper-left light answered by a low amber glow, pronounced tenebrism, soot black, lead gray, oxidized brass, luminous glass, layered oil glazes and restrained impasto; no people, no readable text.",
#     },
#     {
#         "id": "electronic_magnetic_materials",
#         "science": "electronic, optical and magnetic materials; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide lateral Baroque arrangement of broad concentric arcs on a polished slate shelf; a cobalt silicon wafer tilted against a low brass rest; an enamelled copper coil encircling a dark horseshoe magnet; translucent calcite balanced by stepped ferrite tiles; a short fiber-optic strand releasing a few pale points; dark teal silk falling in monumental folds, cool light gathering into warm copper reflections, sculptural chiaroscuro, mineral surfaces, broad brushwork and glazed highlights; no people, no readable text.",
#     },
#     {
#         "id": "mechanics_ocean_aerospace_control",
#         "science": "mechanics and computational mechanics; ocean engineering; aerospace, electrical and control systems engineering",
#         "prompt": "RIJKSOIL, a medium-wide Baroque workshop composition swept by a wing-shaped diagonal above a shallow pewter basin; a brass gyroscope inside its circular gimbal, a small airfoil raised on pins, a steel gear crossed by calipers, a copper-wound servo coupled to a feedback pendulum, and a carved wave crest beside a rolled salt-stained chart; storm-blue canvas and charcoal wool form large shadowed planes; hard left light traces rivets, wet pewter, scratched steel and oil-dark brass with vigorous loaded brushwork; no people, no readable text.",
#     },
#     {
#         "id": "manufacturing_networks",
#         "science": "industrial and manufacturing engineering; computer networks and communications",
#         "prompt": "RIJKSOIL, a medium-wide low Baroque composition carrying a chain of mechanisms across an oil-darkened cast-iron plate; an articulated gripper poised over a precision gear train ending in a polished bearing; a punched brass card joined to woven copper cable; cream ceramic signal insulators rhythmically crossing the rear edge; a heavy brown curtain billows into cavernous shadow while a high left glint breaks across steel, oily brass, woven wire and chalky ceramic, coarse impasto and monumental repetition; no people, no readable text.",
#     },
#     {
#         "id": "mathematics_computation",
#         "science": "mathematics; computational theory; geometry and topology",
#         "prompt": "RIJKSOIL, a medium-wide cabinet-like scholarly Baroque still life unfolding around open brass compasses on a broad chalk-dusted slate; a wooden polyhedron, faint non-readable geometric diagrams, ivory counting rods, exposed calculator wheels and a dark topology loop over folded graph parchment; moss-green baize and aged parchment form quiet vertical layers; angled candlelight, measured geometry, slate black, ivory, worn brass and wood, contemplative chiaroscuro and softly glazed surfaces; no people, no readable text.",
#     },
#     {
#         "id": "computer_science_ai_vision",
#         "science": "computer science; artificial intelligence; computer vision and pattern recognition; information systems",
#         "prompt": "RIJKSOIL, a medium-wide symmetrical Baroque nocturne built around an antique camera lens like a mechanical eye; layered cobalt circuit boards rise behind its toothed blackened-brass housing; cream punched cards meet a glass field of restrained square lights while branching gold conductors spread across the lower plane; a cool square illumination from the left balances one warm copper gleam, lacquer blue, amber glass, centralized drama, deep glazing and luminous accents; no people, no readable text.",
#     },
#     {
#         "id": "operations_economics",
#         "science": "management science and operations research; economics and econometrics; accounting",
#         "prompt": "RIJKSOIL, a medium-wide Dutch Golden Age merchant-table composition ascending from coin stacks to a brass balance beam; an oxblood leather ledger lies open on a shallow writing slope beside a dark abacus, cargo miniatures, a clear sand timer and folded sheets bearing non-readable curves; tobacco-brown drapery gathers into one generous fold; warm candlelight multiplies across tarnished silver, copper, rubbed leather, paper and dark wood in pyramidal order and sober chiaroscuro; no people, no readable text.",
#     },
#     {
#         "id": "strategy_politics_relations",
#         "science": "strategy and management; political science and international relations",
#         "prompt": "RIJKSOIL, a medium-wide courtly Baroque still life leading opposing ebony and ivory chess pieces toward a small terrestrial globe; a brass compass opens over an unreadable coastal chart on a cherrywood campaign box; treaty ribbons, red sealing wax and restrained crimson threads connect colored map pins; dark carmine damask swells behind the globe, theatrical left candlelight catches wax, silk and brass, dramatic diagonals and sumptuous glazing; no people, no readable text.",
#     },
#     {
#         "id": "sociology_philosophy",
#         "science": "sociology; political science; philosophy, knowledge and ethics",
#         "prompt": "RIJKSOIL, a medium-wide civic vanitas arranged around a shallow pewter bowl of voting tokens on a cracked black-marble ledge; clustered wooden figures of varied heights stand among census tally sticks, three linked rings and an open illegible leather book weighted by a river stone; a dark convex mirror and small brass balance catch one severe beeswax candle; smoke-gray linen and olive velvet descend into enveloping shadow, worn wood, fibrous paper, dull pewter and grave translucent glazes; no people, no readable text.",
#     },
#     {
#         "id": "psychology_cognitive_science",
#         "science": "clinical and social psychology; psychiatry and mental health; cognitive neuroscience",
#         "prompt": "RIJKSOIL, a medium-wide asymmetrical Baroque arrangement orbiting a pale ivory wax brain and a reflected theatrical mask; a wooden maze aligns with a slender metronome, ambiguous ink cards scatter among memory beads, and a silver tuning fork crosses the foreground; plum felt, pale maple and a dusky violet curtain open onto a narrow black recess; soft divided light, theatrical doubling, velvety shadows and layered oil color; no people, no readable text.",
#     },
#     {
#         "id": "public_environmental_health",
#         "science": "public, environmental and occupational health; epidemiology; general health professions",
#         "prompt": "RIJKSOIL, a medium-wide field-kit Baroque still life spreading practical instruments in a calm arc from an opened galvanized case; a brass air-sampling pump and pleated filter, a small respirator, worn leather glove, clear water vial, silver thermometer and an epidemiological map with colored pins but no labels; deep green canvas rises behind them with dust and one water stain; clear left window light reveals particles across metal, fabric and glass, earthy realism and weighty forms; no people, no readable text.",
#     },
#     {
#         "id": "neuroscience_physiology_cardiovascular",
#         "science": "neuroscience and neurology; physiology; endocrinology, diabetes and metabolism; cardiology and cardiovascular medicine",
#         "prompt": "RIJKSOIL, a medium-wide anatomical Baroque arc joining an ivory wax brain to refined wax models of a heart and paired lungs; a delicate electrode crown sends red and blue nerve threads toward a coiled brass stethoscope, a clear insulin vial, a reflex hammer and a ruby pulse watch; indigo cloth crosses a burgundy leather case under warm silver light, non-gory sculptural modeling, deep recession, luminous glass and humane layered oil glazes; no people, no readable text.",
#     },
#     {
#         "id": "oncology_immunity_pathology",
#         "science": "cancer research and oncology; hematology; immunology; pathology and forensic medicine",
#         "prompt": "RIJKSOIL, a medium-wide non-gory laboratory vanitas rising from a ruby glass dish toward an angled brass microscope; translucent red droplets, pathology slides, branching ivory antibody forms and pale cell spheres gather around a closed black specimen box; clear glass rests over dark crimson cloth against a black-burgundy recess; sharp left light turns the microscope rim gold and the slides luminous, cavernous shadow, transparent glazes and precise impasto; no people, no readable text.",
#     },
#     {
#         "id": "genetics_evolution_ecology",
#         "science": "molecular and cell biology; genetics; infectious diseases; evolution, ecology, behavior, food and plant science",
#         "prompt": "RIJKSOIL, a medium-wide naturalist's Baroque crescent sweeping from a glass double helix and abstract petri colonies toward a fossil ammonite, spiral shells, a pressed fern, seed pods and a sliced heritage pear; translucent cell vesicles mingle with a pale finch skull and dark beetle on a weathered sandstone shelf; forest-brown and green-black drapery frames cool glass and autumnal fruit, tactile bone, ribbed shell, leaf, seed and moist flesh in layered glazes; no people, no readable text.",
#     },
#     {
#         "id": "toxicology_chemistry_sustainable_materials",
#         "science": "health, toxicology and mutagenesis; chemistry and spectroscopy; biomaterials; polymers; water science; renewable energy and sustainability; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide vertical alchemical Baroque still life rising through a coiled glass alembic with amber reagent drops and descending across a spectroscopy prism, charred leaf, clear polymer film, porous biomaterial mesh, blue solar cell, copper battery plate and pure-water vial; pale soapstone bears old amber rings beneath burnt-orange fabric and a tar-black wall; firelit left illumination refracts through glass and oxidized copper, rich layered paint and fiery chiaroscuro; no people, no readable text.",
#     },
# ]


## 3. GPU, repository, and compatible dependencies

This uses the same proven Colab environment as the pairwise notebook.
A dependency install requests one kernel restart; subsequent healthy
imports do not reinstall.


In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

print({"python": sys.version, "platform": platform.platform()})
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch is missing; use a Colab GPU runtime and rerun this cell.") from error
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required.")
print({"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda})

project_path = Path(PROJECT_ROOT)
if not (project_path / "pyproject.toml").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, PROJECT_ROOT])
elif UPDATE_REPOSITORY:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])

core_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import numpy, scipy, transformers, pydantic; from diffusers import Flux2KleinPipeline",
    ],
    capture_output=True,
    text=True,
)
if core_probe.returncode != 0:
    print("Installing the notebook's pinned FLUX environment because the clean import probe failed:")
    print(core_probe.stderr[-2000:])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r",
        str(project_path / "requirements-colab.txt"),
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
    raise RuntimeError(
        "Dependencies installed successfully. Restart the notebook kernel once, then rerun from section 1."
    )

import importlib
package_source = str(project_path / "src")
if package_source not in sys.path:
    sys.path.insert(0, package_source)
importlib.invalidate_caches()
import flowmorph_klein
from diffusers import Flux2KleinPipeline

project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_ROOT, "rev-parse", "HEAD"], text=True
).strip()
print({
    "repository_commit": project_commit,
    "flowmorph_source": flowmorph_klein.__file__,
})


## 4. Mount Drive and reserve or resume a numbered run

No OpenAI key is needed: prompt conditioning is splined directly from
the editable anchor prompts.


In [ ]:
import json
import re
from datetime import datetime, timezone

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", PROJECT_NAME):
    raise ValueError("PROJECT_NAME may contain only letters, numbers, underscores, and hyphens")

DRIVE_ENABLED = False
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("Drive mounting requires a Google Colab kernel.") from error
    drive.mount("/content/drive")
    drive_base = Path(DRIVE_PROJECT_BASE)
    drive_base.mkdir(parents=True, exist_ok=True)
    DRIVE_ENABLED = True
else:
    drive_base = None

def reserve_numbered_run(parent, project_name):
    project_root = Path(parent) / project_name
    project_root.mkdir(parents=True, exist_ok=True)
    numbers = []
    prefix = f"{project_name}_"
    for candidate in project_root.iterdir():
        if candidate.is_dir() and candidate.name.startswith(prefix):
            token = candidate.name[len(prefix):].split("_", 1)[0]
            if token.isdigit():
                numbers.append(int(token))
    sequence = max(numbers, default=0) + 1
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    while True:
        candidate = project_root / f"{project_name}_{sequence:04d}_{timestamp}"
        try:
            candidate.mkdir(parents=False, exist_ok=False)
        except FileExistsError:
            sequence += 1
            continue
        return candidate

if RESUME_RUN_DIRECTORY is not None:
    RUN_DIRECTORY = Path(RESUME_RUN_DIRECTORY).expanduser()
    if not RUN_DIRECTORY.is_dir():
        raise FileNotFoundError(f"RESUME_RUN_DIRECTORY does not exist: {RUN_DIRECTORY}")
elif DRIVE_ENABLED:
    RUN_DIRECTORY = reserve_numbered_run(drive_base, PROJECT_NAME)
else:
    RUN_DIRECTORY = reserve_numbered_run(LOCAL_ASSET_ROOT, PROJECT_NAME)

for child in (
    "base_frames", "trials", "previews", "video", "metadata",
    "periodic_spline", "diagnostics",
):
    (RUN_DIRECTORY / child).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

run_identity = {
    "project": PROJECT_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "persistent": DRIVE_ENABLED,
    "run_directory": str(RUN_DIRECTORY),
    "interpolation": "periodic_cubic_bspline_through_fitted_flowmorph_endpoints",
    "terminal_duplicate": False,
}
(RUN_DIRECTORY / "metadata" / "run_identity.json").write_text(
    json.dumps(run_identity, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("Run directory:", RUN_DIRECTORY)
print("Every generated image and manifest is written here as soon as it exists.")


## 5. Validate the circular spline contract and preview cost

At least four anchors are required. The reported frame count is the
unique pre-RIFE frame count; the opening anchor is not repeated at the
end.


In [ ]:
if BASE_PROMPT_COUNT is None:
    BASE_PROMPT_COUNT = len(BASE_STAGES)
elif not 4 <= BASE_PROMPT_COUNT <= len(BASE_STAGES):
    raise ValueError(f"BASE_PROMPT_COUNT must be between 4 and {len(BASE_STAGES)}")
if BASE_PROMPT_COUNT < 4:
    raise ValueError("A periodic cubic B-spline needs at least four anchors")
if not (256 <= IMAGE_WIDTH <= 2048 and IMAGE_WIDTH % 16 == 0):
    raise ValueError("IMAGE_WIDTH must be 256–2048 and divisible by 16")
if not (256 <= IMAGE_HEIGHT <= 2048 and IMAGE_HEIGHT % 16 == 0):
    raise ValueError("IMAGE_HEIGHT must be 256–2048 and divisible by 16")
if not 1 <= IMAGE_INFERENCE_STEPS <= 100:
    raise ValueError("IMAGE_INFERENCE_STEPS must be between 1 and 100")
if not 0 <= IMAGE_GUIDANCE_SCALE <= 20:
    raise ValueError("IMAGE_GUIDANCE_SCALE must be between 0 and 20")
if not 0 < IMAGE_LORA_SCALE <= 4:
    raise ValueError("IMAGE_LORA_SCALE must lie in (0, 4]")
if not 0 <= BASE_REFERENCE_GRAIN_STRENGTH <= 0.25:
    raise ValueError("BASE_REFERENCE_GRAIN_STRENGTH must lie in [0, 0.25]")
if not 0 < BASE_REFERENCE_DENOISE_STRENGTH <= 1:
    raise ValueError("BASE_REFERENCE_DENOISE_STRENGTH must lie in (0, 1]")
if not 32 <= FLUX_PROMPT_MAX_SEQUENCE_LENGTH <= 512:
    raise ValueError("FLUX_PROMPT_MAX_SEQUENCE_LENGTH must lie in [32, 512]")
if FLOWMORPH_START_TIMESTEP_INDEX != FLOWMORPH_RENDER_INDICES[0]:
    raise ValueError("The first render index must equal the FlowMorph start index")
if FLOWMORPH_RENDER_INDICES != sorted(set(FLOWMORPH_RENDER_INDICES)):
    raise ValueError("FLOWMORPH_RENDER_INDICES must be strictly increasing")
if FLOWMORPH_RENDER_INDICES[-1] >= FLOWMORPH_SCHEDULER_POINTS:
    raise ValueError("FLOWMORPH_RENDER_INDICES must be smaller than scheduler points")
if FLOWMORPH_SOURCE_OPTIMIZATION_STEPS != FLOWMORPH_TARGET_OPTIMIZATION_STEPS:
    raise ValueError("Cached endpoints require one shared optimization-step count")
if FLOWMORPH_SOURCE_OPTIMIZATION_STEPS < 1:
    raise ValueError("FlowMorph optimization steps must be positive")
if FLOWMORPH_FIT_LORA_SCALE != IMAGE_LORA_SCALE:
    raise ValueError("FlowMorph fit LoRA scale must match IMAGE_LORA_SCALE")
if FLOWMORPH_RENDER_LORA_SCALE != IMAGE_LORA_SCALE:
    raise ValueError("FlowMorph render LoRA scale must match IMAGE_LORA_SCALE")
if FLOWMORPH_GUIDANCE_SCALE != IMAGE_GUIDANCE_SCALE:
    raise ValueError("FlowMorph guidance must match IMAGE_GUIDANCE_SCALE")
for name, value in {
    "FLOWMORPH_ENDPOINT_BATCH_SIZE": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "FLOWMORPH_RENDER_BATCH_SIZE": FLOWMORPH_RENDER_BATCH_SIZE,
    "FLOWMORPH_DECODE_BATCH_SIZE": FLOWMORPH_DECODE_BATCH_SIZE,
    "SPLINE_STREAM_CHUNK_SIZE": SPLINE_STREAM_CHUNK_SIZE,
}.items():
    if value < 1:
        raise ValueError(f"{name} must be positive")
if FLOWMORPH_CFG_EXECUTION not in {"sequential", "batched"}:
    raise ValueError("FLOWMORPH_CFG_EXECUTION must be sequential or batched")
if SPLINE_MIN_FRAMES_PER_SEGMENT < 2:
    raise ValueError("SPLINE_MIN_FRAMES_PER_SEGMENT must be at least 2")
if SPLINE_FRAMES_PER_ANCHOR < SPLINE_MIN_FRAMES_PER_SEGMENT:
    raise ValueError("SPLINE_FRAMES_PER_ANCHOR is below the per-segment minimum")
if SPLINE_PREVIEW_FRAMES_PER_ANCHOR < 2:
    raise ValueError("SPLINE_PREVIEW_FRAMES_PER_ANCHOR must be at least 2")
if not 0 <= SPLINE_TIMING_DISTANCE_STRENGTH <= 1:
    raise ValueError("SPLINE_TIMING_DISTANCE_STRENGTH must lie in [0, 1]")
if not 0 < SPLINE_TIMING_DISTANCE_EXPONENT <= 1:
    raise ValueError("SPLINE_TIMING_DISTANCE_EXPONENT must lie in (0, 1]")
if not 1 <= SPLINE_TIMING_MAX_SEGMENT_RATIO <= 3:
    raise ValueError("SPLINE_TIMING_MAX_SEGMENT_RATIO must lie in [1, 3]")
if TEMPORAL_TONE_WINDOW_RADIUS < 1:
    raise ValueError("TEMPORAL_TONE_WINDOW_RADIUS must be positive")
if not 0 <= TEMPORAL_TONE_STRENGTH <= 1:
    raise ValueError("TEMPORAL_TONE_STRENGTH must lie in [0, 1]")
if VIDEO_SLOWDOWN_FACTOR < 1:
    raise ValueError("VIDEO_SLOWDOWN_FACTOR must be at least 1")

ACTIVE_BASE_STAGES = BASE_STAGES[:BASE_PROMPT_COUNT]
ids = [item["id"] for item in ACTIVE_BASE_STAGES]
if len(ids) != len(set(ids)) or any(not re.fullmatch(r"[a-z0-9_]+", item) for item in ids):
    raise ValueError("Anchor IDs must be unique lowercase snake_case values")
for item in ACTIVE_BASE_STAGES:
    if not item["science"].strip() or not item["prompt"].strip():
        raise ValueError(f"Blank science or prompt in {item['id']}")
    if item["prompt"].casefold().count(LORA_TRIGGER.casefold()) != 1:
        raise ValueError(f"{item['id']} must contain the LoRA trigger exactly once")

SPLINE_TOTAL_FRAMES = BASE_PROMPT_COUNT * SPLINE_FRAMES_PER_ANCHOR
print({
    "anchor_images": BASE_PROMPT_COUNT,
    "unique_endpoint_fits": BASE_PROMPT_COUNT,
    "periodic_spline_frames": SPLINE_TOTAL_FRAMES,
    "terminal_duplicate": False,
    "continuity_at_seam": "C2 (position, velocity, and curvature)",
    "timing_distance_strength": SPLINE_TIMING_DISTANCE_STRENGTH,
    "maximum_segment_ratio": SPLINE_TIMING_MAX_SEGMENT_RATIO,
    "rife_multiplier": RIFE_MULTIPLIER,
})
print("Circular anchor order:", " → ".join(ids), "→", ids[0])


## 6. Load RIJKSOIL and optionally test one random anchor

Use this quick image to tune LoRA, guidance, dimensions, and inference
steps before generating the complete anchor cycle.


In [ ]:
import gc
import os
import random
import shutil
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from PIL import Image, ImageFilter
from flowmorph_klein.lora import load_flux2_lora
from flowmorph_klein.trajectory import prepare_flux2_klein_img2img_inputs

try:
    import peft.tuners.lora.torchao as peft_torchao_dispatch
except ImportError:
    peft_torchao_dispatch = None
else:
    peft_torchao_dispatch.is_torchao_available = lambda: False

downloaded_lora = Path(hf_hub_download(
    repo_id=LORA_SOURCE,
    filename=LORA_WEIGHT_NAME,
    revision=LORA_REVISION,
    cache_dir=HF_CACHE_DIR,
))
lora_stage_directory = Path(HF_CACHE_DIR) / "flowmorph_lora_files" / LORA_REVISION[:12]
lora_stage_directory.mkdir(parents=True, exist_ok=True)
LOCAL_LORA_PATH = lora_stage_directory / LORA_WEIGHT_NAME
if not LOCAL_LORA_PATH.is_file():
    try:
        os.link(downloaded_lora.resolve(), LOCAL_LORA_PATH)
    except OSError:
        shutil.copy2(downloaded_lora, LOCAL_LORA_PATH)
if LOCAL_LORA_PATH.stat().st_size != downloaded_lora.stat().st_size:
    raise RuntimeError(f"Staged LoRA size mismatch at {LOCAL_LORA_PATH}")

def release_flux_pipeline():
    previous = globals().pop("FLUX_PIPE", None)
    globals().pop("FLUX_PIPE_LORA_SCALE", None)
    if previous is not None:
        maybe_free = getattr(previous, "maybe_free_model_hooks", None)
        if callable(maybe_free):
            maybe_free()
        del previous
        gc.collect()
        torch.cuda.empty_cache()

def load_flux_pipeline():
    pipeline = Flux2KleinPipeline.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    report = load_flux2_lora(
        pipeline,
        str(LOCAL_LORA_PATH),
        adapter_name=LORA_ADAPTER_NAME,
        scale=IMAGE_LORA_SCALE,
        require_base_9b_provenance=False,
        allow_distilled_9b=True,
    )
    pipeline.fuse_lora(
        components=["transformer"],
        lora_scale=1.0,
        safe_fusing=True,
        adapter_names=[LORA_ADAPTER_NAME],
    )
    pipeline.unload_lora_weights()
    remaining = [
        name for name, _ in pipeline.transformer.named_parameters()
        if "lora_" in name.casefold() or ".lora" in name.casefold()
    ]
    if remaining:
        raise RuntimeError("LoRA fusion left runtime parameters: " + ", ".join(remaining[:5]))
    pipeline.enable_model_cpu_offload()
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()
    return pipeline, report

if "FLUX_PIPE" in globals() and globals().get("FLUX_PIPE_LORA_SCALE") != float(IMAGE_LORA_SCALE):
    print("LoRA scale changed; rebuilding the fused pipeline.")
    release_flux_pipeline()
if "FLUX_PIPE" not in globals():
    FLUX_PIPE, LORA_REPORT = load_flux_pipeline()
    FLUX_PIPE_LORA_SCALE = float(IMAGE_LORA_SCALE)
    print("Loaded a device-safe fused-LoRA pipeline.")
else:
    print("Reusing the fused pipeline at the current LoRA scale.")


FLUX_PROMPT_TOKENIZER = FLUX_PIPE.tokenizer

def flux_prompt_token_count(prompt):
    messages = [{"role": "user", "content": prompt}]
    templated = FLUX_PROMPT_TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = FLUX_PROMPT_TOKENIZER(
        templated,
        add_special_tokens=False,
        truncation=False,
    )
    return len(encoded["input_ids"])

def validate_flux_prompt_length(prompt, label="Prompt"):
    token_count = flux_prompt_token_count(prompt)
    if token_count > FLUX_PROMPT_MAX_SEQUENCE_LENGTH:
        raise ValueError(
            f"{label} tokenizes to {token_count} tokens after the FLUX chat "
            f"template; maximum is {FLUX_PROMPT_MAX_SEQUENCE_LENGTH}"
        )
    return token_count

if RUN_TRIAL_KEYFRAME:
    system_random = random.SystemRandom()
    trial_index = (
        TRIAL_KEYFRAME_INDEX
        if TRIAL_KEYFRAME_INDEX is not None
        else system_random.randrange(len(ACTIVE_BASE_STAGES))
    )
    if not 0 <= trial_index < len(ACTIVE_BASE_STAGES):
        raise IndexError("TRIAL_KEYFRAME_INDEX is outside the active anchor range")
    trial_seed = TRIAL_SEED if TRIAL_SEED is not None else system_random.randrange(2**31)
    trial_stage = ACTIVE_BASE_STAGES[trial_index]
    validate_flux_prompt_length(trial_stage["prompt"], "Trial anchor prompt")
    trial_result = FLUX_PIPE(
        prompt=trial_stage["prompt"],
        height=IMAGE_HEIGHT,
        width=IMAGE_WIDTH,
        num_inference_steps=IMAGE_INFERENCE_STEPS,
        guidance_scale=IMAGE_GUIDANCE_SCALE,
        generator=torch.Generator(device="cuda").manual_seed(trial_seed),
        output_type="pil",
        max_sequence_length=FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    )
    trial_image = trial_result.images[0].convert("RGB")
    trial_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    trial_directory = RUN_DIRECTORY / "trials" / f"{trial_stamp}_{trial_stage['id']}_{trial_seed}"
    trial_directory.mkdir(parents=True, exist_ok=False)
    trial_path = trial_directory / "trial.png"
    trial_image.save(trial_path)
    (trial_directory / "settings.json").write_text(json.dumps({
        "stage": trial_stage,
        "seed": trial_seed,
        "lora_scale": IMAGE_LORA_SCALE,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "inference_steps": IMAGE_INFERENCE_STEPS,
        "size": [IMAGE_WIDTH, IMAGE_HEIGHT],
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    preview = trial_image.copy()
    preview.thumbnail((TRIAL_DISPLAY_MAX_WIDTH, TRIAL_DISPLAY_MAX_WIDTH))
    display(Markdown(f"### Trial anchor: `{trial_stage['id']}`"))
    display(preview)
    print({"path": str(trial_path), "seed": trial_seed, "prompt_index": trial_index})
    del trial_result, trial_image, preview
else:
    print("Trial skipped.")


## 7. Generate prompt-only circular anchor paintings

The first is text-to-image. Later anchors may use a blurred, grained
previous image as an ordinary latent img2img start. No flat beige
canvas, masks, or post-compositing are involved.


In [ ]:
from flowmorph_klein.art_loop import make_soft_reference

BASE_DIRECTORY = RUN_DIRECTORY / "base_frames"
REFERENCE_DIRECTORY = BASE_DIRECTORY / "soft_references"
BASE_MANIFEST_PATH = RUN_DIRECTORY / "metadata" / "base_manifest.json"
BASE_RECORDS = []

def generate_prompt_anchor(prompt, seed, reference=None):
    validate_flux_prompt_length(prompt, "Anchor generation prompt")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    kwargs = {
        "prompt": prompt,
        "height": IMAGE_HEIGHT,
        "width": IMAGE_WIDTH,
        "num_inference_steps": IMAGE_INFERENCE_STEPS,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "generator": generator,
        "output_type": "pil",
        "max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    }
    generation_report = {
        "mode": "text_to_image",
        "requested_img2img_strength": None,
        "effective_start_sigma": None,
    }
    if reference is not None:
        generation_inputs = prepare_flux2_klein_img2img_inputs(
            FLUX_PIPE,
            reference,
            width=IMAGE_WIDTH,
            height=IMAGE_HEIGHT,
            num_inference_steps=IMAGE_INFERENCE_STEPS,
            strength=BASE_REFERENCE_DENOISE_STRENGTH,
            generator=generator,
        )
        kwargs["sigmas"] = list(generation_inputs.sigmas)
        kwargs["latents"] = generation_inputs.latents
        generation_report = {
            "mode": "latent_img2img_from_weak_previous_reference",
            "requested_img2img_strength": (
                generation_inputs.requested_strength
            ),
            "effective_start_sigma": generation_inputs.effective_start_sigma,
            "denoising_steps": generation_inputs.denoising_steps,
        }
    result = FLUX_PIPE(**kwargs)
    if not result.images:
        raise RuntimeError("FLUX returned no anchor image")
    return result.images[0].convert("RGB"), generation_report

if not REGENERATE_BASE_FRAMES and BASE_MANIFEST_PATH.is_file():
    BASE_RECORDS = json.loads(
        BASE_MANIFEST_PATH.read_text(encoding="utf-8")
    )["records"]
    missing = [
        item["path"]
        for item in BASE_RECORDS
        if not Path(item["path"]).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing resumed anchor images: " + ", ".join(missing)
        )
    resumed_contract = [
        (record["uid"], record["science"], record["prompt"])
        for record in BASE_RECORDS
    ]
    current_contract = [
        (f"base_{index:03d}", stage["science"], stage["prompt"])
        for index, stage in enumerate(ACTIVE_BASE_STAGES)
    ]
    if resumed_contract != current_contract:
        raise RuntimeError(
            "Editable anchor prompts differ from the saved anchors. "
            "Regenerate or resume the matching run."
        )
    print(f"Loaded {len(BASE_RECORDS)} existing anchor records.")
else:
    BASE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    previous = None
    for index, stage in enumerate(ACTIVE_BASE_STAGES):
        seed = BASE_SEED + index
        reference = None
        reference_path = None
        if previous is not None and BASE_CONTINUITY_ENABLED:
            reference = make_soft_reference(
                previous,
                # A blend of 1.0 means 100% blurred previous image. No
                # fixed beige/gray background canvas contributes.
                reference_blend=1.0,
                blur_radius=BASE_REFERENCE_BLUR,
                grain_strength=BASE_REFERENCE_GRAIN_STRENGTH,
                grain_seed=seed,
            )
            if SAVE_SOFT_REFERENCES:
                REFERENCE_DIRECTORY.mkdir(parents=True, exist_ok=True)
                reference_path = (
                    REFERENCE_DIRECTORY / f"reference_{index:03d}.png"
                )
                reference.save(reference_path, format="PNG", compress_level=4)
        image, generation_report = generate_prompt_anchor(
            stage["prompt"],
            seed,
            reference=reference,
        )
        output_path = BASE_DIRECTORY / f"{index:03d}_{stage['id']}.png"
        image.save(output_path, format="PNG", compress_level=4)
        record = {
            "uid": f"base_{index:03d}",
            "kind": "base",
            "round": 0,
            "science": stage["science"],
            "prompt": stage["prompt"],
            "generation_prompt": stage["prompt"],
            "generation_prompt_token_count": validate_flux_prompt_length(
                stage["prompt"],
                "Saved anchor prompt",
            ),
            "seed": seed,
            "path": str(output_path),
            "soft_reference_path": (
                str(reference_path) if reference_path else None
            ),
            "base_continuity_used": reference is not None,
            "base_reference_source": (
                "blurred_grained_previous_without_flat_canvas"
            ),
            "base_reference_blur": BASE_REFERENCE_BLUR,
            "base_reference_grain_strength": (
                BASE_REFERENCE_GRAIN_STRENGTH
            ),
            "generation_mode": generation_report["mode"],
            "img2img_strength": generation_report[
                "requested_img2img_strength"
            ],
            "effective_start_sigma": generation_report[
                "effective_start_sigma"
            ],
        }
        BASE_RECORDS.append(record)
        BASE_MANIFEST_PATH.write_text(json.dumps({
            "project": PROJECT_NAME,
            "complete": len(BASE_RECORDS) == len(ACTIVE_BASE_STAGES),
            "records": BASE_RECORDS,
        }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        if previous is not None:
            previous.close()
        previous = image.copy()
        image.close()
        if reference is not None:
            reference.close()
        print(
            f"Anchor {index + 1}/{len(ACTIVE_BASE_STAGES)} saved: "
            f"{output_path.name}"
        )
    if previous is not None:
        previous.close()

if len(BASE_RECORDS) != len(ACTIVE_BASE_STAGES):
    raise RuntimeError("The anchor manifest is incomplete.")
print(f"Prepared {len(BASE_RECORDS)} cyclic anchors in {BASE_DIRECTORY}")


In [ ]:
from flowmorph_klein.visualization import make_contact_sheet

base_contact_sheet_path = RUN_DIRECTORY / "previews" / "base_contact_sheet.png"
base_images = [Image.open(item["path"]).convert("RGB") for item in BASE_RECORDS]
make_contact_sheet(
    base_images,
    base_contact_sheet_path,
    columns=min(CONTACT_SHEET_COLUMNS, len(base_images)),
    labels=[item["uid"] for item in BASE_RECORDS],
)
for image in base_images:
    image.close()
base_preview = Image.open(base_contact_sheet_path).convert("RGB")
base_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### Anchor paintings — compact contact sheet"))
display(base_preview)
del base_preview, base_images
print("Full-resolution anchors and contact sheet:", BASE_DIRECTORY)

saved_reference_paths = [
    Path(item["soft_reference_path"])
    for item in BASE_RECORDS
    if item.get("soft_reference_path") and Path(item["soft_reference_path"]).is_file()
]
if saved_reference_paths:
    reference_contact_sheet_path = (
        RUN_DIRECTORY / "previews" / "anchor_soft_reference_contact_sheet.png"
    )
    reference_images = []
    for path in saved_reference_paths:
        with Image.open(path) as opened:
            thumbnail = opened.convert("RGB")
            thumbnail.thumbnail((192, 192))
            reference_images.append(thumbnail)
    make_contact_sheet(
        reference_images,
        reference_contact_sheet_path,
        columns=min(CONTACT_SHEET_COLUMNS, len(reference_images)),
        labels=[path.stem for path in saved_reference_paths],
    )
    for image in reference_images:
        image.close()
    reference_preview = Image.open(reference_contact_sheet_path).convert("RGB")
    reference_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### Blurred/grained anchor initialization images"))
    display(reference_preview)
    reference_preview.close()
    print("Full-resolution anchor initialization images:", saved_reference_paths[0].parent)


## 8. Estimate restrained nonuniform timing around the complete loop

Thumbnail color/gradient distance is only a timing proxy. Square-root
tempering, a uniform blend, and a hard ratio cap prevent extreme
dwell-time changes. The closing last→first distance is included.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from flowmorph_klein.spline_trajectory import (
    PeriodicCubicBSplineBasis,
    allocate_periodic_segment_frames,
    periodic_thumbnail_distances,
    regularized_periodic_timing,
    sample_periodic_timeline,
)

SPLINE_RAW_DISTANCES = periodic_thumbnail_distances(
    [record["path"] for record in BASE_RECORDS],
    analysis_size=SPLINE_DISTANCE_ANALYSIS_SIZE,
    color_weight=SPLINE_DISTANCE_COLOR_WEIGHT,
)
SPLINE_TIMING = regularized_periodic_timing(
    SPLINE_RAW_DISTANCES,
    distance_strength=SPLINE_TIMING_DISTANCE_STRENGTH,
    distance_exponent=SPLINE_TIMING_DISTANCE_EXPONENT,
    maximum_segment_ratio=SPLINE_TIMING_MAX_SEGMENT_RATIO,
)
SPLINE_SEGMENT_FRAME_COUNTS = allocate_periodic_segment_frames(
    SPLINE_TIMING.segment_durations,
    total_frames=SPLINE_TOTAL_FRAMES,
    minimum_frames_per_segment=SPLINE_MIN_FRAMES_PER_SEGMENT,
)
SPLINE_SAMPLES = sample_periodic_timeline(
    SPLINE_TIMING,
    SPLINE_SEGMENT_FRAME_COUNTS,
)
SPLINE_BASIS = PeriodicCubicBSplineBasis(SPLINE_TIMING.knot_times)
knot_weights = SPLINE_BASIS.weights(SPLINE_TIMING.knot_times[:-1])
if not np.allclose(
    knot_weights,
    np.eye(len(BASE_RECORDS)),
    atol=1e-9,
    rtol=0,
):
    raise RuntimeError("Periodic spline does not interpolate every anchor exactly")
for derivative in (0, 1, 2):
    seam_values = SPLINE_BASIS.weights(
        (0.0, 1.0),
        derivative=derivative,
    )
    if not np.allclose(
        seam_values[0],
        seam_values[1],
        atol=1e-9,
        rtol=0,
    ):
        raise RuntimeError(
            f"Periodic spline derivative {derivative} is discontinuous at the seam"
        )

timing_records = []
for index, record in enumerate(BASE_RECORDS):
    timing_records.append({
        "segment": index,
        "left_uid": record["uid"],
        "right_uid": BASE_RECORDS[(index + 1) % len(BASE_RECORDS)]["uid"],
        "raw_visual_distance": SPLINE_RAW_DISTANCES[index],
        "regularized_duration": SPLINE_TIMING.segment_durations[index],
        "frame_count": SPLINE_SEGMENT_FRAME_COUNTS[index],
        "knot_time": SPLINE_TIMING.knot_times[index],
    })
SPLINE_TIMING_MANIFEST = RUN_DIRECTORY / "metadata" / "periodic_spline_timing.json"
SPLINE_TIMING_MANIFEST.write_text(json.dumps({
    "method": "regularized visual-distance periodic timing",
    "distance_strength": SPLINE_TIMING_DISTANCE_STRENGTH,
    "distance_exponent": SPLINE_TIMING_DISTANCE_EXPONENT,
    "maximum_segment_ratio": SPLINE_TIMING_MAX_SEGMENT_RATIO,
    "total_unique_frames": len(SPLINE_SAMPLES),
    "terminal_duplicate": False,
    "segments": timing_records,
}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

figure, axis = plt.subplots(figsize=(max(9, len(BASE_RECORDS) * 0.7), 3.5))
axis.bar(
    range(len(BASE_RECORDS)),
    SPLINE_SEGMENT_FRAME_COUNTS,
    color="#536f87",
)
axis.set_xticks(range(len(BASE_RECORDS)))
axis.set_xticklabels(
    [record["uid"] for record in BASE_RECORDS],
    rotation=55,
    ha="right",
)
axis.set_ylabel("unique frames in outgoing segment")
axis.set_title("Restrained nonuniform periodic timing")
axis.grid(axis="y", alpha=0.2)
figure.tight_layout()
SPLINE_TIMING_PLOT = RUN_DIRECTORY / "previews" / "periodic_spline_timing.png"
figure.savefig(SPLINE_TIMING_PLOT, dpi=160, facecolor="white")
plt.close(figure)
timing_preview = Image.open(SPLINE_TIMING_PLOT).convert("RGB")
timing_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(timing_preview)
timing_preview.close()
print({
    "timing_manifest": str(SPLINE_TIMING_MANIFEST),
    "segment_frame_counts": SPLINE_SEGMENT_FRAME_COUNTS,
    "duration_ratio": round(
        max(SPLINE_TIMING.segment_durations)
        / min(SPLINE_TIMING.segment_durations),
        4,
    ),
    "exact_anchor_knots": len(BASE_RECORDS),
    "runtime_seam_audit": "C2 passed",
    "terminal_duplicate": False,
})


## 9. Fit every unique endpoint once and decode canonical knots

One model remains loaded for the entire sequence and the backward
preflight runs once. Each image is fitted once even though the global
curve enters and leaves it. The decoded reconstruction is the exact
knot used by the final sequence.


In [ ]:
import gc
import hashlib
import torch
from flowmorph_klein.cli import select_hardware_profile
from flowmorph_klein.config import ProjectTemplateConfig, load_config, resolve_config
from flowmorph_klein.pipeline import FlowMorphRunner
from flowmorph_klein.sequence import FlowMorphSequenceSession, SequenceEndpointRequest

def validate_sequence_flowmorph_contract(config):
    for name, value in (("width", config.input.width), ("height", config.input.height)):
        if not 256 <= value <= 2048 or value % 16 != 0:
            raise ValueError(f"input.{name} must be 256–2048 and divisible by 16")
    if config.flowmorph.frame_count < 3:
        raise ValueError("The bootstrap FlowMorph config needs at least three prompt slots")
    if config.flowmorph.render_conditioning_mode.value != "prompt_schedule":
        raise ValueError("Sequence FlowMorph requires prompt_schedule conditioning")
    if len(config.input.bridge_prompts or ()) != config.flowmorph.frame_count:
        raise ValueError("Bootstrap prompt schedule length must equal frame_count")

ProjectTemplateConfig._validate_full_shape_contract = validate_sequence_flowmorph_contract
print("Isolated periodic-spline FlowMorph contract enabled.")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def stable_fingerprint(payload):
    serialized = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()

BASE_PROMPT_TOKEN_COUNTS = {}
for record in BASE_RECORDS:
    BASE_PROMPT_TOKEN_COUNTS[record["uid"]] = {
        "prompt": validate_flux_prompt_length(
            record["prompt"],
            f"{record['uid']} FlowMorph endpoint prompt",
        ),
        "generation_prompt": validate_flux_prompt_length(
            record.get("generation_prompt", record["prompt"]),
            f"{record['uid']} anchor generation prompt",
        ),
    }
print({
    "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    "anchor_prompt_token_counts": BASE_PROMPT_TOKEN_COUNTS,
})

# Release the fused anchor generator. One differentiable model is loaded below
# and retained for all endpoint fits and every spline render chunk.
release_flux_pipeline()
SEQUENCE_ROOT = RUN_DIRECTORY / "flowmorph_sequence"
SEQUENCE_SESSION_CONTRACT = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "lora_sha256": file_sha256(LOCAL_LORA_PATH),
    "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
    "render_lora_scale": FLOWMORPH_RENDER_LORA_SCALE,
    "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
    "scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
    "start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
    "optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
    "u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
    "render_indices": list(FLOWMORPH_RENDER_INDICES),
    "width": IMAGE_WIDTH,
    "height": IMAGE_HEIGHT,
    "conditioning": "periodic_cubic_bspline_anchor_embeddings",
    "flux_prompt_max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
    "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
    "cfg_execution": FLOWMORPH_CFG_EXECUTION,
    "batch_oom_backoff": FLOWMORPH_BATCH_OOM_BACKOFF,
}
SEQUENCE_SESSION_FINGERPRINT = stable_fingerprint(SEQUENCE_SESSION_CONTRACT)
SEQUENCE_SESSION_DIRECTORY = (
    SEQUENCE_ROOT / "sessions" / f"session_{SEQUENCE_SESSION_FINGERPRINT[:12]}"
)
SEQUENCE_ENDPOINT_ROOT = SEQUENCE_ROOT / "endpoints"
SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT = (
    SEQUENCE_ROOT / "endpoint_reconstructions"
)
SEQUENCE_ASSET_ROOT = SEQUENCE_ROOT / "encoded_inputs"
for directory in (
    SEQUENCE_ROOT,
    SEQUENCE_ENDPOINT_ROOT,
    SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT,
    SEQUENCE_ASSET_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

bootstrap_left = BASE_RECORDS[0]
bootstrap_right = BASE_RECORDS[1]
bootstrap_schedule = [
    bootstrap_left["prompt"],
    bootstrap_left["prompt"],
    bootstrap_right["prompt"],
]
session_overrides = {
    "run_mode": "experimental",
    "project.name": f"{PROJECT_NAME}_sequence_session",
    "model.id": MODEL_ID,
    "model.revision": MODEL_REVISION,
    "lora.source": str(LOCAL_LORA_PATH),
    "lora.revision": None,
    "lora.weight_name": LOCAL_LORA_PATH.name,
    "lora.adapter_name": LORA_ADAPTER_NAME,
    "lora.fit_scale": FLOWMORPH_FIT_LORA_SCALE,
    "lora.render_scale": FLOWMORPH_RENDER_LORA_SCALE,
    "lora.require_base_9b_compatibility": False,
    "lora.allow_distilled_9b": True,
    "input.source_image": str(bootstrap_left["path"]),
    "input.target_image": str(bootstrap_right["path"]),
    "input.source_prompt": bootstrap_left["prompt"],
    "input.target_prompt": bootstrap_right["prompt"],
    "input.bridge_prompt": None,
    "input.bridge_prompts": bootstrap_schedule,
    "input.width": IMAGE_WIDTH,
    "input.height": IMAGE_HEIGHT,
    "flowmorph.scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
    "flowmorph.start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
    "flowmorph.optimization_steps_source": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "flowmorph.optimization_steps_target": FLOWMORPH_TARGET_OPTIMIZATION_STEPS,
    "flowmorph.pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
    "flowmorph.u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
    "flowmorph.frame_count": len(bootstrap_schedule),
    "flowmorph.render_indices": FLOWMORPH_RENDER_INDICES,
    "flowmorph.alpha_schedule": "linear",
    "flowmorph.render_conditioning_mode": "prompt_schedule",
    "flowmorph.checkpoint_every": FLOWMORPH_CHECKPOINT_EVERY,
    "guidance.scale": FLOWMORPH_GUIDANCE_SCALE,
    "reproducibility.seed": BASE_SEED,
    "paths.input_root": str(RUN_DIRECTORY),
    "paths.work_root": str(
        Path(LOCAL_ASSET_ROOT)
        / PROJECT_NAME
        / "sequence_work"
        / SEQUENCE_SESSION_FINGERPRINT[:12]
    ),
    "paths.result_root": str(SEQUENCE_ROOT),
    "paths.hf_cache": HF_CACHE_DIR,
    "paths.drive_root": None,
    "output.fps": int(SOURCE_SEQUENCE_FPS),
    "output.save_contact_sheet": False,
    "output.save_webp": False,
    "output.save_gif": False,
    "output.save_mp4": False,
    # Required by the reusable config validator. The sequence session does
    # not call FlowMorphRunner.run(), so no research archive is created.
    "output.create_zip": True,
}
session_template = load_config(CONFIG_PATH, overrides=session_overrides)
session_profile = select_hardware_profile(
    PROFILE if PROFILE != "auto" else session_template.model.profile
)
session_config = resolve_config(
    session_template,
    selected_profile=session_profile,
    check_input_files=True,
)
session_resume = (
    RESUME_FLOWMORPH_SEQUENCE
    and (SEQUENCE_SESSION_DIRECTORY / "run_manifest.json").is_file()
)
SEQUENCE_RUNNER = FlowMorphRunner.from_config(
    session_config,
    run_directory=SEQUENCE_SESSION_DIRECTORY,
)
SEQUENCE_RUNNER.prepare(resume=session_resume)
SEQUENCE_SESSION = FlowMorphSequenceSession(
    SEQUENCE_RUNNER,
    render_batch_size=FLOWMORPH_RENDER_BATCH_SIZE,
    decode_batch_size=FLOWMORPH_DECODE_BATCH_SIZE,
    cfg_execution=FLOWMORPH_CFG_EXECUTION,
    oom_backoff=FLOWMORPH_BATCH_OOM_BACKOFF,
)
PROBE_REPORT = SEQUENCE_SESSION.run_backward_probe_once()
print({
    "model_loads": 1,
    "backward_probes": 1,
    "fit_steps_per_unique_endpoint": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
    "probe_peak_reserved_gib": round(PROBE_REPORT.peak_reserved_vram_bytes / (1024 ** 3), 3),
    "endpoint_batch_size": FLOWMORPH_ENDPOINT_BATCH_SIZE,
    "render_batch_size": FLOWMORPH_RENDER_BATCH_SIZE,
    "decode_batch_size": FLOWMORPH_DECODE_BATCH_SIZE,
    "cfg_execution": FLOWMORPH_CFG_EXECUTION,
})

IMAGE_ASSET_CACHE, PROMPT_CONDITIONING_CACHE = SEQUENCE_SESSION.seed_prepared_assets(
    bootstrap_left["uid"],
    bootstrap_right["uid"],
)
ENDPOINT_CACHE = {}
ENDPOINT_FINGERPRINTS = {}
ENDPOINT_RECONSTRUCTION_PATHS = {}
UNIQUE_ENDPOINT_FIT_COUNT = 0

def endpoint_fingerprint(record):
    return stable_fingerprint({
        "uid": record["uid"],
        "image_sha256": file_sha256(record["path"]),
        "prompt": record["prompt"],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "lora_sha256": file_sha256(LOCAL_LORA_PATH),
        "fit_lora_scale": FLOWMORPH_FIT_LORA_SCALE,
        "guidance_scale": FLOWMORPH_GUIDANCE_SCALE,
        "scheduler_points": FLOWMORPH_SCHEDULER_POINTS,
        "start_timestep_index": FLOWMORPH_START_TIMESTEP_INDEX,
        "optimization_steps": FLOWMORPH_SOURCE_OPTIMIZATION_STEPS,
        "pred_learning_rate": FLOWMORPH_PRED_LEARNING_RATE,
        "u_learning_rate": FLOWMORPH_U_LEARNING_RATE,
        "width": IMAGE_WIDTH,
        "height": IMAGE_HEIGHT,
    })

def ensure_sequence_assets(records, prompts=()):
    missing_prompts = [
        prompt for prompt in prompts
        if prompt not in PROMPT_CONDITIONING_CACHE
    ]
    missing_images = {
        record["uid"]: (
            record["path"],
            SEQUENCE_ASSET_ROOT / f"{record['uid']}.png",
        )
        for record in records
        if record["uid"] not in IMAGE_ASSET_CACHE
    }
    record_prompts = [
        record["prompt"] for record in records
        if record["prompt"] not in PROMPT_CONDITIONING_CACHE
    ]
    for prompt in [*record_prompts, *missing_prompts]:
        validate_flux_prompt_length(prompt, "FlowMorph conditioning prompt")
    if record_prompts or missing_prompts or missing_images:
        new_prompts, new_images = SEQUENCE_SESSION.encode_missing_assets(
            prompts=[*record_prompts, *missing_prompts],
            images=missing_images,
        )
        PROMPT_CONDITIONING_CACHE.update(new_prompts)
        IMAGE_ASSET_CACHE.update(new_images)


def endpoint_reconstruction_path(record):
    fingerprint = endpoint_fingerprint(record)
    return (
        SEQUENCE_ENDPOINT_RECONSTRUCTION_ROOT
        / f"{record['uid']}_{fingerprint[:12]}.png"
    )


def register_existing_endpoint_reconstructions(records):
    for record in records:
        path = endpoint_reconstruction_path(record)
        if path.is_file():
            ENDPOINT_RECONSTRUCTION_PATHS[record["uid"]] = path
            ENDPOINT_FINGERPRINTS.setdefault(
                record["uid"],
                endpoint_fingerprint(record),
            )


def ensure_endpoint_reconstructions(records, progress_label):
    unique_records = {
        record["uid"]: record
        for record in records
        if record["uid"] in ENDPOINT_CACHE
    }
    missing_records = []
    missing_paths = []
    for uid, record in unique_records.items():
        path = endpoint_reconstruction_path(record)
        if path.is_file():
            ENDPOINT_RECONSTRUCTION_PATHS[uid] = path
            continue
        missing_records.append(record)
        missing_paths.append(path)
    if not missing_records:
        return
    frames = SEQUENCE_SESSION.render_endpoint_reconstructions(
        endpoints=[
            ENDPOINT_CACHE[record["uid"]]
            for record in missing_records
        ],
        conditionings=[
            PROMPT_CONDITIONING_CACHE[record["prompt"]]
            for record in missing_records
        ],
    )
    SEQUENCE_SESSION.decode_frames_to_paths(frames, missing_paths)
    for record, path in zip(
        missing_records,
        missing_paths,
        strict=True,
    ):
        ENDPOINT_RECONSTRUCTION_PATHS[record["uid"]] = path
        print(
            f"{progress_label}: canonical endpoint "
            f"{record['uid']} -> {path.name}"
        )
    del frames


def fit_sequence_endpoints(records, progress_label):
    global UNIQUE_ENDPOINT_FIT_COUNT
    register_existing_endpoint_reconstructions(records)
    unique_records = {
        record["uid"]: record
        for record in records
        if record["uid"] not in ENDPOINT_CACHE
    }
    if unique_records:
        requests = []
        fingerprints = {}
        for uid, record in unique_records.items():
            fingerprint = endpoint_fingerprint(record)
            fingerprints[uid] = fingerprint
            checkpoint_directory = (
                SEQUENCE_ENDPOINT_ROOT
                / f"{uid}_{fingerprint[:12]}"
            )
            requests.append(SequenceEndpointRequest(
                endpoint_key=uid,
                asset=IMAGE_ASSET_CACHE[uid],
                conditioning=(
                    PROMPT_CONDITIONING_CACHE[record["prompt"]]
                ),
                checkpoint_directory=checkpoint_directory,
                resume=(
                    RESUME_FLOWMORPH_SEQUENCE
                    and checkpoint_directory.exists()
                ),
            ))
        fitted = SEQUENCE_SESSION.fit_endpoints(
            requests,
            batch_size=FLOWMORPH_ENDPOINT_BATCH_SIZE,
        )
        for uid, result in fitted.items():
            ENDPOINT_CACHE[uid] = result.endpoint
            ENDPOINT_FINGERPRINTS[uid] = fingerprints[uid]
            UNIQUE_ENDPOINT_FIT_COUNT += 1
            print(
                f"{progress_label}: {uid}; "
                f"steps={result.completed_steps}; "
                f"checkpoint_reused={result.resumed}"
            )
    ensure_endpoint_reconstructions(records, progress_label)


def fit_sequence_endpoint(record, progress_label):
    fit_sequence_endpoints([record], progress_label)
    return ENDPOINT_CACHE[record["uid"]]


ensure_sequence_assets(BASE_RECORDS)
fit_sequence_endpoints(BASE_RECORDS, "Periodic endpoint fit")
if len(ENDPOINT_CACHE) != len(BASE_RECORDS):
    raise RuntimeError("Not every unique anchor endpoint is fitted")
if len(ENDPOINT_RECONSTRUCTION_PATHS) != len(BASE_RECORDS):
    raise RuntimeError("Not every fitted endpoint has a canonical reconstruction")

fit_summary = {
    "unique_endpoint_count": len(ENDPOINT_CACHE),
    "new_endpoint_fits_this_session": UNIQUE_ENDPOINT_FIT_COUNT,
    "canonical_reconstructions": {
        uid: str(path)
        for uid, path in ENDPOINT_RECONSTRUCTION_PATHS.items()
    },
    "session_contract": SEQUENCE_SESSION_CONTRACT,
    "session_fingerprint": SEQUENCE_SESSION_FINGERPRINT,
}
(RUN_DIRECTORY / "metadata" / "periodic_endpoint_fit_summary.json").write_text(
    json.dumps(fit_summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print({
    "unique_endpoints_ready": len(ENDPOINT_CACHE),
    "fitted_only_once_per_unique_image": True,
    "canonical_endpoint_reconstructions": len(ENDPOINT_RECONSTRUCTION_PATHS),
    "model_loads": 1,
    "backward_probes": 1,
})


## 10. Coarse global spline quality gate

This renders one interior sample in every circular segment, including
the closing last→first segment. It tests the actual global state and
prompt-embedding spline before the full render.


In [ ]:
from flowmorph_klein.spline_trajectory import (
    PeriodicConditioningSpline,
    PeriodicFlowMorphSpline,
    PeriodicSplineFlowMorphRenderer,
)

SPLINE_ENDPOINTS = [ENDPOINT_CACHE[record["uid"]] for record in BASE_RECORDS]
SPLINE_CONDITIONINGS = [
    PROMPT_CONDITIONING_CACHE[record["prompt"]]
    for record in BASE_RECORDS
]
SPLINE_STATE_TRAJECTORY = PeriodicFlowMorphSpline(
    SPLINE_ENDPOINTS,
    SPLINE_BASIS,
)
SPLINE_PROMPT_TRAJECTORY = PeriodicConditioningSpline(
    SPLINE_CONDITIONINGS,
    SPLINE_BASIS,
)
SPLINE_RENDERER = PeriodicSplineFlowMorphRenderer(
    SEQUENCE_SESSION,
    SPLINE_STATE_TRAJECTORY,
    SPLINE_PROMPT_TRAJECTORY,
)

if RUN_SPLINE_COARSE_PREVIEW:
    preview_counts = tuple(
        SPLINE_PREVIEW_FRAMES_PER_ANCHOR for _ in BASE_RECORDS
    )
    preview_samples = sample_periodic_timeline(SPLINE_TIMING, preview_counts)
    preview_fingerprint = stable_fingerprint({
        "session_fingerprint": SEQUENCE_SESSION_FINGERPRINT,
        "endpoint_fingerprints": [
            ENDPOINT_FINGERPRINTS[record["uid"]]
            for record in BASE_RECORDS
        ],
        "sample_times": [item.time for item in preview_samples],
    })
    preview_directory = (
        RUN_DIRECTORY
        / "trials"
        / f"periodic_spline_coarse_{preview_fingerprint[:12]}"
    )
    preview_directory.mkdir(parents=True, exist_ok=True)
    preview_records = []
    interior_samples = [
        item for item in preview_samples if item.anchor_index is None
    ]
    interior_paths = [
        preview_directory / f"frame_{item.frame_index:05d}.png"
        for item in interior_samples
    ]
    missing = [
        (sample, path)
        for sample, path in zip(interior_samples, interior_paths, strict=True)
        if not path.is_file()
    ]
    if missing:
        frames = SPLINE_RENDERER.render([item.time for item, _ in missing])
        SEQUENCE_SESSION.decode_frames_to_paths(
            frames,
            [path for _, path in missing],
        )
        del frames
    interior_lookup = {
        sample.frame_index: path
        for sample, path in zip(interior_samples, interior_paths, strict=True)
    }
    for sample in preview_samples:
        if sample.anchor_index is not None:
            anchor = BASE_RECORDS[sample.anchor_index]
            path = ENDPOINT_RECONSTRUCTION_PATHS[anchor["uid"]]
            label = f"{anchor['uid']} exact knot"
        else:
            path = interior_lookup[sample.frame_index]
            label = (
                f"segment {sample.segment_index} "
                f"{sample.segment_fraction:.2f}"
            )
        preview_records.append({"path": str(path), "label": label})
    thumbnails = []
    for item in preview_records:
        with Image.open(item["path"]) as opened:
            thumbnail = opened.convert("RGB")
            thumbnail.thumbnail((256, 256))
            thumbnails.append(thumbnail)
    preview_sheet = preview_directory / "periodic_spline_coarse_sheet.png"
    make_contact_sheet(
        thumbnails,
        preview_sheet,
        columns=4,
        labels=[item["label"] for item in preview_records],
    )
    for image in thumbnails:
        image.close()
    shown = Image.open(preview_sheet).convert("RGB")
    shown.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### Coarse global periodic spline quality gate"))
    display(shown)
    shown.close()
    print({
        "preview_sheet": str(preview_sheet),
        "preview_unique_frames": len(preview_records),
        "includes_last_to_first_segment": True,
        "terminal_duplicate": False,
    })
else:
    print("Coarse periodic spline preview skipped.")


### Full periodic B-spline FlowMorph render

Frames stream to Drive in bounded chunks. Rerunning with an unchanged
render fingerprint reuses completed PNGs. Exact anchor knots use their
canonical fitted reconstructions and are never rerendered.


In [ ]:
SPLINE_RENDER_CONTRACT = {
    "session_fingerprint": SEQUENCE_SESSION_FINGERPRINT,
    "endpoint_fingerprints": [
        ENDPOINT_FINGERPRINTS[record["uid"]]
        for record in BASE_RECORDS
    ],
    "knot_times": list(SPLINE_TIMING.knot_times),
    "segment_frame_counts": list(SPLINE_SEGMENT_FRAME_COUNTS),
    "sample_times": [item.time for item in SPLINE_SAMPLES],
    "state_curve": "periodic interpolating cubic B-spline over z, delta, and direction/log-magnitude u",
    "conditioning_curve": "same periodic cubic B-spline over anchor prompt embeddings",
    "seam_continuity": "C2",
    "terminal_duplicate": False,
}
SPLINE_RENDER_FINGERPRINT = stable_fingerprint(SPLINE_RENDER_CONTRACT)
SPLINE_FRAME_DIRECTORY = (
    RUN_DIRECTORY
    / "periodic_spline"
    / f"frames_{SPLINE_RENDER_FINGERPRINT[:12]}"
)
SPLINE_FRAME_DIRECTORY.mkdir(parents=True, exist_ok=True)
FINAL_SEQUENCE_MANIFEST = (
    RUN_DIRECTORY / "metadata" / "final_periodic_bspline_flowmorph_sequence.json"
)

interior_jobs = []
FINAL_RECORDS = []
for sample in SPLINE_SAMPLES:
    if sample.anchor_index is not None:
        anchor = BASE_RECORDS[sample.anchor_index]
        frame_path = ENDPOINT_RECONSTRUCTION_PATHS[anchor["uid"]]
        kind = "canonical_fitted_anchor"
        anchor_uid = anchor["uid"]
    else:
        frame_path = (
            SPLINE_FRAME_DIRECTORY
            / f"{sample.frame_index:07d}_t{sample.time:.10f}.png"
        )
        kind = "periodic_bspline_interior"
        anchor_uid = None
        if not (
            SPLINE_REUSE_RENDERED_FRAMES
            and frame_path.is_file()
        ):
            interior_jobs.append((sample, frame_path))
    FINAL_RECORDS.append({
        "uid": f"spline_{sample.frame_index:07d}",
        "kind": kind,
        "path": str(frame_path),
        "time": sample.time,
        "segment_index": sample.segment_index,
        "segment_fraction": sample.segment_fraction,
        "anchor_uid": anchor_uid,
        "render_fingerprint": SPLINE_RENDER_FINGERPRINT,
    })

for start in range(0, len(interior_jobs), SPLINE_STREAM_CHUNK_SIZE):
    chunk = interior_jobs[start:start + SPLINE_STREAM_CHUNK_SIZE]
    frames = SPLINE_RENDERER.render([item.time for item, _ in chunk])
    paths = [path for _, path in chunk]
    SEQUENCE_SESSION.decode_frames_to_paths(frames, paths)
    del frames
    completed = min(start + len(chunk), len(interior_jobs))
    print(
        f"Periodic spline interiors rendered: {completed}/"
        f"{len(interior_jobs)}; latest={paths[-1].name}",
        flush=True,
    )

missing_final_paths = [
    item["path"] for item in FINAL_RECORDS
    if not Path(item["path"]).is_file()
]
if missing_final_paths:
    raise FileNotFoundError(
        "Periodic render is incomplete; first missing path: "
        + missing_final_paths[0]
    )

final_payload = {
    "project": PROJECT_NAME,
    "method": "global periodic cubic B-spline through fitted FlowMorph endpoints",
    "render_contract": SPLINE_RENDER_CONTRACT,
    "render_fingerprint": SPLINE_RENDER_FINGERPRINT,
    "final_count": len(FINAL_RECORDS),
    "anchor_count": len(BASE_RECORDS),
    "interior_count": sum(
        item["kind"] == "periodic_bspline_interior"
        for item in FINAL_RECORDS
    ),
    "exact_canonical_anchor_knots": True,
    "seam_continuity": "C2",
    "terminal_duplicate": False,
    "records": FINAL_RECORDS,
}
FINAL_SEQUENCE_MANIFEST.write_text(
    json.dumps(final_payload, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print({
    "final_manifest": str(FINAL_SEQUENCE_MANIFEST),
    "unique_frames": len(FINAL_RECORDS),
    "new_interiors_rendered": len(interior_jobs),
    "canonical_anchor_reuses": len(BASE_RECORDS),
    "smooth_last_to_first_segment": True,
    "terminal_duplicate": False,
})

# RIFE needs the GPU next. The PNGs and manifest are already safely on Drive.
for name in (
    "SPLINE_RENDERER",
    "SPLINE_STATE_TRAJECTORY",
    "SPLINE_PROMPT_TRAJECTORY",
    "SPLINE_ENDPOINTS",
    "SPLINE_CONDITIONINGS",
    "ENDPOINT_CACHE",
    "IMAGE_ASSET_CACHE",
    "PROMPT_CONDITIONING_CACHE",
    "SEQUENCE_SESSION",
    "SEQUENCE_RUNNER",
):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print("Released the periodic FlowMorph model; RIFE can now use the GPU.")


## 11. Stabilize, preview, and audit the circular spline sequence

The raw output is already a closed curve: there is no special final pair and no
duplicated first frame. Every fitted anchor appears exactly once. Optional
temporal tone stabilization writes corrected copies only; raw spline PNGs remain
untouched. The quiet-cut rotation changes only where playback begins.


In [ ]:
import json
import imageio_ffmpeg
import numpy as np
import shutil
import subprocess
import tempfile
from pathlib import Path
from PIL import Image
from IPython.display import Markdown, Video, display
from flowmorph_klein.temporal_tone import (
    TemporalToneConfig,
    stabilize_cyclic_tone,
)
from flowmorph_klein.visualization import make_contact_sheet

# A Colab reconnect clears Python variables while completed manifests and
# images remain on Drive. Recover the final sequence before auditing it.
if "FINAL_RECORDS" not in globals():
    if "RUN_DIRECTORY" not in globals():
        raise RuntimeError(
            "RUN_DIRECTORY is not initialized. Set RESUME_RUN_DIRECTORY to the "
            "completed Drive run, then rerun the setup cells before section 10."
        )
    restored_manifest_candidates = []
    explicit_manifest = globals().get("FINAL_SEQUENCE_MANIFEST")
    if explicit_manifest:
        restored_manifest_candidates.append(Path(explicit_manifest))
    restored_manifest_candidates.extend([
        RUN_DIRECTORY
        / "metadata"
        / "final_periodic_bspline_flowmorph_sequence_tone_stabilized.json",
        RUN_DIRECTORY / "metadata" / "final_periodic_bspline_flowmorph_sequence.json",
        RUN_DIRECTORY / "metadata" / "final_recursive_sequence.json",
    ])
    restored_manifest_path = next(
        (path for path in restored_manifest_candidates if path.is_file()),
        None,
    )
    if restored_manifest_path is None:
        discovered_manifests = sorted(
            [
                *RUN_DIRECTORY.parent.glob(
                    "*/metadata/final_periodic_bspline_flowmorph_sequence_tone_stabilized.json"
                ),
                *RUN_DIRECTORY.parent.glob(
                    "*/metadata/final_periodic_bspline_flowmorph_sequence.json"
                ),
                *RUN_DIRECTORY.parent.glob("*/metadata/final_recursive_sequence.json"),
            ],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        discovery_note = (
            " Completed manifests in sibling runs: "
            + ", ".join(str(path.parent.parent) for path in discovered_manifests[:5])
            if discovered_manifests
            else " No completed sibling-run manifest was found."
        )
        raise RuntimeError(
            "FINAL_RECORDS is not in memory and the saved sequence manifest was "
            f"not found in {RUN_DIRECTORY / 'metadata'}. Set RESUME_RUN_DIRECTORY "
            "to the completed Drive run and rerun the setup cells." + discovery_note
        )
    restored_payload = json.loads(restored_manifest_path.read_text(encoding="utf-8"))
    FINAL_RECORDS = restored_payload["records"]
    FINAL_SEQUENCE_MANIFEST = restored_manifest_path
    print({
        "restored_final_sequence": True,
        "frames": len(FINAL_RECORDS),
        "manifest": str(FINAL_SEQUENCE_MANIFEST),
    })


raw_final_records = []
for item in FINAL_RECORDS:
    raw_item = dict(item)
    raw_item["path"] = item.get("raw_flowmorph_path", item["path"])
    raw_final_records.append(raw_item)

if TEMPORAL_TONE_STABILIZATION_ENABLED:
    tone_directory = RUN_DIRECTORY / "temporal_tone_stabilization"
    raw_sequence_manifest = (
        RUN_DIRECTORY / "metadata" / "final_periodic_bspline_flowmorph_sequence.json"
    )
    if not raw_sequence_manifest.is_file():
        raw_sequence_manifest = Path(FINAL_SEQUENCE_MANIFEST)
    tone_result = stabilize_cyclic_tone(
        [item["path"] for item in raw_final_records],
        tone_directory / "corrected_frames",
        config=TemporalToneConfig(
            window_radius=TEMPORAL_TONE_WINDOW_RADIUS,
            strength=TEMPORAL_TONE_STRENGTH,
            mean_threshold=TEMPORAL_TONE_MEAN_THRESHOLD,
            contrast_threshold=TEMPORAL_TONE_CONTRAST_THRESHOLD,
            mad_multiplier=TEMPORAL_TONE_MAD_MULTIPLIER,
            max_mean_shift=TEMPORAL_TONE_MAX_MEAN_SHIFT,
            max_contrast_scale_delta=(
                TEMPORAL_TONE_MAX_CONTRAST_SCALE_DELTA
            ),
            analysis_max_side=TEMPORAL_TONE_ANALYSIS_MAX_SIDE,
        ),
        report_path=tone_directory / "temporal_tone_report.json",
        reuse_existing=TEMPORAL_TONE_REUSE_EXISTING,
    )
    stabilized_records = []
    for item, raw_item, stabilized_path, frame_audit in zip(
        FINAL_RECORDS,
        raw_final_records,
        tone_result.output_paths,
        tone_result.report["frames"],
        strict=True,
    ):
        stabilized = dict(item)
        stabilized["raw_flowmorph_path"] = raw_item["path"]
        stabilized["path"] = str(stabilized_path)
        stabilized["temporal_tone_corrected"] = frame_audit["corrected"]
        stabilized["temporal_tone_report_path"] = str(tone_result.report_path)
        stabilized_records.append(stabilized)
    FINAL_RECORDS = stabilized_records
    tone_sequence_manifest = (
        RUN_DIRECTORY
        / "metadata"
        / "final_periodic_bspline_flowmorph_sequence_tone_stabilized.json"
    )
    tone_sequence_manifest.write_text(json.dumps({
        "project": PROJECT_NAME,
        "cyclic": True,
        "source_manifest": str(raw_sequence_manifest),
        "temporal_tone_stabilization_enabled": True,
        "temporal_tone_fingerprint": tone_result.report["fingerprint"],
        "temporal_tone_report": str(tone_result.report_path),
        "corrected_count": tone_result.report["corrected_count"],
        "corrected_indices": tone_result.report["corrected_indices"],
        "cache_hit": tone_result.cache_hit,
        "final_count": len(FINAL_RECORDS),
        "records": FINAL_RECORDS,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    FINAL_SEQUENCE_MANIFEST = tone_sequence_manifest
    print({
        "temporal_tone_stabilization": True,
        "corrected_frames": tone_result.report["corrected_count"],
        "total_frames": len(FINAL_RECORDS),
        "cache_hit": tone_result.cache_hit,
        "raw_frames_preserved": True,
        "report": str(tone_result.report_path),
    })
else:
    FINAL_RECORDS = raw_final_records
    print("Temporal tone stabilization disabled; using raw FlowMorph frames.")

if len(FINAL_RECORDS) < 3:
    raise RuntimeError("A cyclic preview needs at least three images")
if LOOP_PREVIEW_RENDER_MAX_SIDE < 128:
    raise ValueError("LOOP_PREVIEW_RENDER_MAX_SIDE must be at least 128")

canonical_paths = [Path(item["path"]) for item in FINAL_RECORDS]

def metric_array(path, size):
    with Image.open(path) as opened:
        sample = opened.convert("RGB")
        sample.thumbnail((size, size))
        return np.asarray(sample, dtype=np.uint8).copy()

def mean_absolute_delta(left, right):
    difference = left.astype(np.int16) - right.astype(np.int16)
    return float(np.mean(np.abs(difference)) / 255.0)

print(f"Reading {len(canonical_paths)} small seam-analysis thumbnails (not full frames)...")
metric_frames = [metric_array(path, LOOP_SEAM_ANALYSIS_SIZE) for path in canonical_paths]
edge_scores = [
    mean_absolute_delta(metric_frames[index], metric_frames[index - 1])
    for index in range(len(metric_frames))
]
quietest_cut_index = int(np.argmin(edge_scores))
export_cut_index = quietest_cut_index if LOOP_AUTO_ROTATE_TO_QUIETEST_CUT else 0
EXPORT_FRAME_PATHS = canonical_paths[export_cut_index:] + canonical_paths[:export_cut_index]
EXPORT_RECORDS = FINAL_RECORDS[export_cut_index:] + FINAL_RECORDS[:export_cut_index]
export_metrics = metric_frames[export_cut_index:] + metric_frames[:export_cut_index]

seam_delta = mean_absolute_delta(export_metrics[0], export_metrics[-1])
median_delta = float(np.median(edge_scores))
seam_ratio = seam_delta / median_delta if median_delta else 0.0
incoming_motion = export_metrics[0].astype(np.int16) - export_metrics[-1].astype(np.int16)
outgoing_motion = export_metrics[1].astype(np.int16) - export_metrics[0].astype(np.int16)
motion_mismatch = float(np.mean(np.abs(outgoing_motion - incoming_motion)) / 255.0)

preview_directory = RUN_DIRECTORY / "previews" / "generated_loop"
preview_directory.mkdir(parents=True, exist_ok=True)
preview_video_path = preview_directory / "generated_loop_reduced.mp4"
preview_stage = Path(tempfile.mkdtemp(prefix="flowmorph_preview_"))
try:
    for index, source_path in enumerate(EXPORT_FRAME_PATHS):
        staged_path = preview_stage / f"{index:07d}.png"
        try:
            staged_path.symlink_to(source_path.resolve())
        except OSError:
            shutil.copy2(source_path, staged_path)
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    subprocess.check_call([
        ffmpeg, "-y", "-framerate", str(SOURCE_SEQUENCE_FPS),
        "-i", str(preview_stage / "%07d.png"),
        "-vf", (
            f"scale={LOOP_PREVIEW_RENDER_MAX_SIDE}:{LOOP_PREVIEW_RENDER_MAX_SIDE}:"
            "force_original_aspect_ratio=decrease:force_divisible_by=2"
        ),
        "-an", "-c:v", "libx264", "-preset", "veryfast",
        "-crf", "22", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(preview_video_path),
    ])
finally:
    shutil.rmtree(preview_stage, ignore_errors=True)

seam_sheet_path = preview_directory / "seam_audit.png"
seam_images = []
try:
    for path in (EXPORT_FRAME_PATHS[-1], EXPORT_FRAME_PATHS[0], EXPORT_FRAME_PATHS[1]):
        with Image.open(path) as opened:
            seam_images.append(opened.convert("RGB"))
    make_contact_sheet(
        seam_images,
        seam_sheet_path,
        columns=3,
        labels=["last before wrap", "playback start", "first after start"],
    )
finally:
    for image in seam_images:
        image.close()
seam_report = {
    "cyclic": True,
    "duplicate_terminal_frame": False,
    "frame_count": len(EXPORT_FRAME_PATHS),
    "auto_rotate": LOOP_AUTO_ROTATE_TO_QUIETEST_CUT,
    "cut_index": export_cut_index,
    "quietest_cut_index": quietest_cut_index,
    "seam_mean_absolute_delta": seam_delta,
    "median_edge_mean_absolute_delta": median_delta,
    "seam_ratio_to_median": seam_ratio,
    "seam_motion_mismatch": motion_mismatch,
    "ordered_uids": [item["uid"] for item in EXPORT_RECORDS],
}
seam_report_path = preview_directory / "seam_audit.json"
seam_report_path.write_text(json.dumps(seam_report, indent=2) + "\n", encoding="utf-8")

seam_preview = Image.open(seam_sheet_path).convert("RGB")
seam_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### Loop seam: last → playback start → next"))
display(seam_preview)
del seam_preview
display(Markdown("### Generated-image loop before RIFE"))
display(Video(
    str(preview_video_path),
    embed=False,
    width=LOOP_PREVIEW_DISPLAY_WIDTH,
    html_attributes="controls loop muted playsinline",
))
del metric_frames, export_metrics, incoming_motion, outgoing_motion
print({
    "frames": len(EXPORT_FRAME_PATHS),
    "cut_index": export_cut_index,
    "seam_vs_median": round(seam_ratio, 4),
    "motion_mismatch": round(motion_mismatch, 6),
    "preview_video": str(preview_video_path),
})


## 12. Prepare pinned Practical-RIFE and its v4.25 model

RIFE receives the lossless unique spline PNGs plus one temporary copy
of frame zero, solely so it can interpolate the circular wrap edge.


In [ ]:
import subprocess
import zipfile

if not RUN_RIFE_POSTPROCESS:
    print("RIFE post-processing disabled in section 1.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("RIFE requires CUDA")
    if RIFE_MULTIPLIER < 2:
        raise ValueError("RIFE_MULTIPLIER must be at least 2")
    if RIFE_BATCH_SIZE < 1:
        raise ValueError("RIFE_BATCH_SIZE must be positive")
    if RIFE_SCALE not in {0.25, 0.5, 1.0, 2.0, 4.0}:
        raise ValueError("RIFE_SCALE must be 0.25, 0.5, 1.0, 2.0, or 4.0")
    if RIFE_FINAL_FPS <= 0 or SOURCE_SEQUENCE_FPS <= 0:
        raise ValueError("Source and final FPS must be positive")
    if RIFE_FINAL_FPS / SOURCE_SEQUENCE_FPS > RIFE_MULTIPLIER:
        raise ValueError("RIFE_MULTIPLIER must be at least RIFE_FINAL_FPS / SOURCE_SEQUENCE_FPS")

    rife_root = Path(RIFE_ROOT)
    if not (rife_root / ".git").is_dir():
        if rife_root.exists():
            raise RuntimeError(f"RIFE_ROOT exists but is not a Git checkout: {rife_root}")
        subprocess.check_call(["git", "clone", "--filter=blob:none", RIFE_REPOSITORY_URL, str(rife_root)])
    installed_revision = subprocess.check_output(
        ["git", "-C", str(rife_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if installed_revision != RIFE_REPOSITORY_REVISION:
        subprocess.check_call([
            "git", "-C", str(rife_root), "fetch", "--depth", "1",
            "origin", RIFE_REPOSITORY_REVISION,
        ])
        subprocess.check_call([
            "git", "-C", str(rife_root), "checkout", "--detach", RIFE_REPOSITORY_REVISION
        ])
    installed_revision = subprocess.check_output(
        ["git", "-C", str(rife_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if installed_revision != RIFE_REPOSITORY_REVISION:
        raise RuntimeError("Practical-RIFE checkout did not resolve to the pinned revision")

    rife_archive = Path(hf_hub_download(
        repo_id=RIFE_MODEL_REPOSITORY,
        filename=RIFE_MODEL_FILENAME,
        revision=RIFE_MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
    ))
    rife_model_root = Path(HF_CACHE_DIR) / "flowmorph_rife_models" / RIFE_MODEL_FILENAME.removesuffix(".zip")
    candidates = list(rife_model_root.rglob("flownet.pkl")) if rife_model_root.exists() else []
    if not candidates:
        rife_model_root.mkdir(parents=True, exist_ok=True)
        root_resolved = rife_model_root.resolve()
        with zipfile.ZipFile(rife_archive) as archive:
            for member in archive.infolist():
                destination = (rife_model_root / member.filename).resolve()
                if not destination.is_relative_to(root_resolved):
                    raise RuntimeError(f"Unsafe path in RIFE archive: {member.filename}")
            archive.extractall(rife_model_root)
        candidates = list(rife_model_root.rglob("flownet.pkl"))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one RIFE flownet.pkl, found {candidates}")
    RIFE_MODEL_DIRECTORY = candidates[0].parent

    RIFE_RUNNER_SOURCE = r'''
import argparse
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

parser = argparse.ArgumentParser()
parser.add_argument("--repo", required=True)
parser.add_argument("--model", required=True)
parser.add_argument("--input", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--multi", type=int, required=True)
parser.add_argument("--scale", type=float, required=True)
parser.add_argument("--batch-size", type=int, default=1)
parser.add_argument("--fp16", action="store_true")
args = parser.parse_args()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for RIFE")
if args.batch_size < 1:
    raise ValueError("--batch-size must be positive")
torch.cuda.set_device(0)
device = torch.device("cuda:0")
sys.path.insert(0, str(Path(args.model).resolve().parent))
sys.path.insert(0, str(Path(args.repo).resolve()))
import train_log.IFNet_HDv3 as rife_ifnet_module
from train_log.IFNet_HDv3 import IFNet

# Practical-RIFE's pinned warplayer builds its cached sampling grid in
# float32. torch.grid_sample requires input and grid to share a dtype,
# so that implementation fails when the network runs in fp16. Replace
# the function in IFNet's module namespace with a device/dtype-aware
# equivalent while retaining the pinned model and weights.
rife_grid_cache = {}

def dtype_safe_warp(tensor_input, tensor_flow):
    cache_key = (
        str(tensor_flow.device),
        str(tensor_flow.dtype),
        tuple(tensor_flow.shape),
    )
    if cache_key not in rife_grid_cache:
        horizontal = torch.linspace(
            -1.0, 1.0, tensor_flow.shape[3],
            device=tensor_flow.device, dtype=tensor_flow.dtype,
        ).view(1, 1, 1, tensor_flow.shape[3]).expand(
            tensor_flow.shape[0], -1, tensor_flow.shape[2], -1
        )
        vertical = torch.linspace(
            -1.0, 1.0, tensor_flow.shape[2],
            device=tensor_flow.device, dtype=tensor_flow.dtype,
        ).view(1, 1, tensor_flow.shape[2], 1).expand(
            tensor_flow.shape[0], -1, -1, tensor_flow.shape[3]
        )
        rife_grid_cache[cache_key] = torch.cat((horizontal, vertical), dim=1)
    normalized_flow = torch.cat((
        tensor_flow[:, 0:1] / ((tensor_input.shape[3] - 1.0) / 2.0),
        tensor_flow[:, 1:2] / ((tensor_input.shape[2] - 1.0) / 2.0),
    ), dim=1)
    grid = (rife_grid_cache[cache_key] + normalized_flow).permute(0, 2, 3, 1)
    return F.grid_sample(
        tensor_input,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True,
    )

rife_ifnet_module.warp = dtype_safe_warp

input_paths = sorted(Path(args.input).glob("*.png"), key=lambda path: int(path.stem))
if len(input_paths) < 2:
    raise ValueError("RIFE input needs at least two numbered PNG files")
output = Path(args.output)
output.mkdir(parents=True, exist_ok=False)

class InferenceModel:
    def __init__(self, model_directory):
        # Load on CPU first, then move the complete initialized module to
        # one explicit CUDA device. This avoids mixed CPU/CUDA parameters
        # with newer torch/checkpoint combinations.
        self.flownet = IFNet()
        state = torch.load(
            str(Path(model_directory) / "flownet.pkl"), map_location="cpu", weights_only=True
        )
        state = {key.removeprefix("module."): value for key, value in state.items()}
        load_result = self.flownet.load_state_dict(state, strict=False)
        if load_result.missing_keys:
            raise RuntimeError(f"RIFE checkpoint missing keys: {load_result.missing_keys}")
        self.flownet.to(device).eval()

    def inference(self, image0, image1, timestep, scale):
        inputs = torch.cat((image0, image1), dim=1)
        scale_list = [16 / scale, 8 / scale, 4 / scale, 2 / scale, 1 / scale]
        _, _, merged = self.flownet(inputs, timestep, scale_list)
        return merged[-1]

model = InferenceModel(args.model)
if args.fp16:
    model.flownet.half()
model_dtype = next(model.flownet.parameters()).dtype
model_device = next(model.flownet.parameters()).device
if model_device != device:
    raise RuntimeError(f"RIFE model resolved to {model_device}, expected {device}")
print({
    "cuda_device": torch.cuda.get_device_name(device),
    "torch": torch.__version__,
    "model_device": str(model_device),
    "model_dtype": str(model_dtype),
    "input_pairs": len(input_paths) - 1,
}, flush=True)
first_image = Image.open(input_paths[0]).convert("RGB")
height, width = first_image.height, first_image.width
block = max(128, int(128 / args.scale))
padded_height = ((height - 1) // block + 1) * block
padded_width = ((width - 1) // block + 1) * block
padding = (0, padded_width - width, 0, padded_height - height)

def load_tensor(path):
    image = Image.open(path).convert("RGB")
    if image.size != (width, height):
        raise ValueError(f"Mismatched input dimensions at {path}: {image.size}")
    array = np.asarray(image, dtype=np.uint8).copy()
    tensor = torch.from_numpy(array.transpose(2, 0, 1)).unsqueeze(0)
    tensor = tensor.to(device=device, dtype=model_dtype) / 255.0
    return F.pad(tensor, padding)

def save_tensor(tensor, path):
    if tensor.shape[0] != 1:
        raise ValueError("save_tensor expects one image")
    array = (tensor[0, :, :height, :width].float().clamp(0, 1) * 255.0).round().byte()
    array = array.permute(1, 2, 0).cpu().numpy()
    Image.fromarray(array, mode="RGB").save(path, compress_level=4)

pair_count = len(input_paths) - 1
shutil.copy2(input_paths[0], output / "0000000.png")
report_every = max(1, pair_count // 20)
pair_start = 0
active_batch_size = min(args.batch_size, pair_count)
with torch.inference_mode():
    while pair_start < pair_count:
        current_size = min(active_batch_size, pair_count - pair_start)
        pair_indices = list(range(pair_start, pair_start + current_size))
        left = None
        right = None
        try:
            left = torch.cat(
                [load_tensor(input_paths[index]) for index in pair_indices],
                dim=0,
            )
            right = torch.cat(
                [load_tensor(input_paths[index + 1]) for index in pair_indices],
                dim=0,
            )
            for step in range(1, args.multi):
                middle = model.inference(
                    left,
                    right,
                    timestep=step / args.multi,
                    scale=args.scale,
                )
                for offset, pair_index in enumerate(pair_indices):
                    output_index = pair_index * args.multi + step
                    save_tensor(
                        middle[offset : offset + 1],
                        output / f"{output_index:07d}.png",
                    )
            for pair_index in pair_indices:
                output_index = (pair_index + 1) * args.multi
                shutil.copy2(
                    input_paths[pair_index + 1],
                    output / f"{output_index:07d}.png",
                )
        except torch.cuda.OutOfMemoryError:
            if current_size == 1:
                raise
            active_batch_size = max(1, (current_size + 1) // 2)
            del left, right
            torch.cuda.empty_cache()
            print(
                f"RIFE OOM; retrying pair batch with batch_size={active_batch_size}",
                flush=True,
            )
            continue
        pair_start += current_size
        if pair_start % report_every == 0 or pair_start == pair_count:
            print(
                f"RIFE pairs: {pair_start}/{pair_count}; "
                f"active_batch_size={active_batch_size}",
                flush=True,
            )
output_count = pair_count * args.multi + 1
print(f"RIFE complete: {output_count} PNG frames")
'''
    RIFE_RUNNER_PATH = Path(LOCAL_ASSET_ROOT) / PROJECT_NAME / "rife_pair_sequence_runner.py"
    RIFE_RUNNER_PATH.parent.mkdir(parents=True, exist_ok=True)
    RIFE_RUNNER_PATH.write_text(RIFE_RUNNER_SOURCE.strip() + "\n", encoding="utf-8")
    print({
        "rife_revision": installed_revision,
        "model_directory": str(RIFE_MODEL_DIRECTORY),
        "multiplier": RIFE_MULTIPLIER,
        "scale": RIFE_SCALE,
        "batch_size": RIFE_BATCH_SIZE,
        "runner": str(RIFE_RUNNER_PATH),
    })


## 13. Interpolate every circular pair with RIFE

The temporary terminal copy is verified pixel-identical and removed
from the unique dense sequence after the wrap pair is interpolated.


In [ ]:
if RUN_RIFE_POSTPROCESS:
    # A Colab reconnect clears Python variables even though completed images and
    # manifests remain on Drive. Restore the ordered export paths when needed.
    if "EXPORT_FRAME_PATHS" not in globals():
        restored_records = globals().get("FINAL_RECORDS")
        if not restored_records:
            restored_manifest_candidates = []
            explicit_manifest = globals().get("FINAL_SEQUENCE_MANIFEST")
            if explicit_manifest:
                restored_manifest_candidates.append(Path(explicit_manifest))
            restored_manifest_candidates.extend([
                RUN_DIRECTORY / "metadata" / "final_periodic_bspline_flowmorph_sequence.json",
                RUN_DIRECTORY / "metadata" / "final_recursive_sequence.json",
            ])
            restored_manifest_path = next(
                (path for path in restored_manifest_candidates if path.is_file()),
                None,
            )
            if restored_manifest_path is None:
                raise RuntimeError(
                    "The generated sequence is not available in memory and its saved "
                    f"manifest was not found in {RUN_DIRECTORY / 'metadata'}. Set "
                    "RESUME_RUN_DIRECTORY to the completed Drive run and rerun the "
                    "setup/assembly cells before RIFE."
                )
            restored_payload = json.loads(restored_manifest_path.read_text(encoding="utf-8"))
            restored_records = restored_payload["records"]

        restored_cut_index = 0
        restored_seam_report_path = (
            RUN_DIRECTORY / "previews" / "generated_loop" / "seam_audit.json"
        )
        if restored_seam_report_path.is_file():
            restored_seam_report = json.loads(
                restored_seam_report_path.read_text(encoding="utf-8")
            )
            restored_cut_index = int(restored_seam_report.get("cut_index", 0))
        elif LOOP_AUTO_ROTATE_TO_QUIETEST_CUT:
            print(
                "No saved seam report was found; using canonical circular order. "
                "Run section 10 first if you want automatic quietest-cut rotation."
            )

        restored_cut_index %= len(restored_records)
        EXPORT_RECORDS = (
            restored_records[restored_cut_index:] + restored_records[:restored_cut_index]
        )
        EXPORT_FRAME_PATHS = [Path(item["path"]) for item in EXPORT_RECORDS]
        missing_export_paths = [path for path in EXPORT_FRAME_PATHS if not path.is_file()]
        if missing_export_paths:
            raise FileNotFoundError(
                "The restored sequence references missing image files; first missing path: "
                f"{missing_export_paths[0]}"
            )
        print({
            "restored_export_sequence": True,
            "frames": len(EXPORT_FRAME_PATHS),
            "cut_index": restored_cut_index,
        })

    postprocess_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    RIFE_WORK_DIRECTORY = Path(LOCAL_ASSET_ROOT) / PROJECT_NAME / "rife_work" / postprocess_stamp
    RIFE_INPUT_DIRECTORY = RIFE_WORK_DIRECTORY / "cyclic_input"
    RIFE_DENSE_DIRECTORY = RIFE_WORK_DIRECTORY / "dense_frames"
    RIFE_RESULTS_DIRECTORY = RUN_DIRECTORY / "video" / postprocess_stamp
    RIFE_INPUT_DIRECTORY.mkdir(parents=True, exist_ok=False)
    RIFE_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=False)

    for index, source_path in enumerate(EXPORT_FRAME_PATHS):
        shutil.copy2(source_path, RIFE_INPUT_DIRECTORY / f"{index:07d}.png")
    shutil.copy2(
        RIFE_INPUT_DIRECTORY / "0000000.png",
        RIFE_INPUT_DIRECTORY / f"{len(EXPORT_FRAME_PATHS):07d}.png",
    )
    command = [
        sys.executable, "-u", str(RIFE_RUNNER_PATH),
        "--repo", str(rife_root),
        "--model", str(RIFE_MODEL_DIRECTORY),
        "--input", str(RIFE_INPUT_DIRECTORY),
        "--output", str(RIFE_DENSE_DIRECTORY),
        "--multi", str(RIFE_MULTIPLIER),
        "--scale", str(RIFE_SCALE),
        "--batch-size", str(RIFE_BATCH_SIZE),
    ]
    if RIFE_USE_FP16:
        command.append("--fp16")
    print(f"Interpolating {len(EXPORT_FRAME_PATHS)} cyclic pairs at {RIFE_MULTIPLIER}× density...")

    def run_rife(command_to_run, label):
        print(f"RIFE attempt: {label}", flush=True)
        process = subprocess.Popen(
            command_to_run,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        log_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            log_lines.append(line)
            print(line, end="", flush=True)
        return_code = process.wait()
        log_text = "".join(log_lines) or "(RIFE produced no subprocess output)"
        if not log_lines:
            print(log_text)
        return return_code, log_text

    return_code, rife_log = run_rife(
        command,
        "fp16" if "--fp16" in command else "fp32",
    )
    if return_code != 0 and "--fp16" in command and RIFE_RETRY_WITH_FP32:
        print("The fp16 RIFE attempt failed; retrying once in fp32.")
        RIFE_DENSE_DIRECTORY = RIFE_WORK_DIRECTORY / "dense_frames_fp32_retry"
        retry_command = [item for item in command if item != "--fp16"]
        output_index = retry_command.index("--output") + 1
        retry_command[output_index] = str(RIFE_DENSE_DIRECTORY)
        return_code, rife_log = run_rife(retry_command, "fp32 fallback")
    if return_code != 0:
        raise RuntimeError(
            "RIFE failed after the available attempt(s). The complete child-process "
            "traceback is printed immediately above. Last output:\n" + rife_log[-6000:]
        )

    dense_with_duplicate = sorted(
        RIFE_DENSE_DIRECTORY.glob("*.png"), key=lambda path: int(path.stem)
    )
    expected = len(EXPORT_FRAME_PATHS) * RIFE_MULTIPLIER + 1
    if len(dense_with_duplicate) != expected:
        raise RuntimeError(f"RIFE wrote {len(dense_with_duplicate)} frames; expected {expected}")
    with Image.open(dense_with_duplicate[0]) as opened:
        first_array = np.asarray(opened.convert("RGB"))
    with Image.open(dense_with_duplicate[-1]) as opened:
        last_array = np.asarray(opened.convert("RGB"))
    if not np.array_equal(first_array, last_array):
        raise RuntimeError("RIFE terminal image is not pixel-identical to the opening image")
    RIFE_DENSE_PATHS = dense_with_duplicate[:-1]
    print({
        "base_cyclic_frames": len(EXPORT_FRAME_PATHS),
        "dense_unique_frames": len(RIFE_DENSE_PATHS),
        "removed_exact_terminal_duplicate": True,
        "local_work_directory": str(RIFE_WORK_DIRECTORY),
        "persistent_results_directory": str(RIFE_RESULTS_DIRECTORY),
    })


## 14. Circular SSIM motion equalization and final H.264 loop

Circular `1 − SSIM` redistributes playback samples by visible motion.
The exported H.264 still contains no duplicated terminal frame.


In [ ]:
if RUN_RIFE_POSTPROCESS:
    import imageio_ffmpeg
    import matplotlib.pyplot as plt
    from IPython.display import Video
    from skimage.metrics import structural_similarity

    def ssim_luma(path):
        with Image.open(path) as opened:
            gray = opened.convert("L")
            gray.thumbnail((RIFE_SSIM_ANALYSIS_SIZE, RIFE_SSIM_ANALYSIS_SIZE))
            return np.asarray(gray, dtype=np.uint8)

    print(f"Computing circular SSIM weights for {len(RIFE_DENSE_PATHS)} dense frames...")
    dense_luma = [ssim_luma(path) for path in RIFE_DENSE_PATHS]
    circular_ssim = np.asarray([
        structural_similarity(dense_luma[index - 1], dense_luma[index], data_range=255)
        for index in range(len(dense_luma))
    ], dtype=np.float64)
    motion_weights = np.maximum(RIFE_SSIM_WEIGHT_FLOOR, 1.0 - circular_ssim)
    frame_positions = np.zeros(len(RIFE_DENSE_PATHS), dtype=np.float64)
    frame_positions[1:] = np.cumsum(motion_weights[1:])
    total_motion = float(frame_positions[-1] + motion_weights[0])

    canonical_duration = len(EXPORT_FRAME_PATHS) / float(SOURCE_SEQUENCE_FPS)
    target_frame_count = int(round(canonical_duration * RIFE_FINAL_FPS))
    if target_frame_count < 3:
        raise ValueError("Final video needs at least three frames")
    if target_frame_count > len(RIFE_DENSE_PATHS):
        raise ValueError("Increase RIFE_MULTIPLIER or lower RIFE_FINAL_FPS")
    targets = np.linspace(0.0, total_motion, target_frame_count, endpoint=False)

    selected_indices = []
    previous_index = -1
    dense_count = len(RIFE_DENSE_PATHS)
    for order, target in enumerate(targets):
        minimum = previous_index + 1
        maximum = dense_count - (target_frame_count - order)
        insertion = int(np.searchsorted(frame_positions, target, side="left"))
        candidates = {
            min(max(insertion, minimum), maximum),
            min(max(insertion - 1, minimum), maximum),
        }
        chosen = min(candidates, key=lambda index: abs(frame_positions[index] - target))
        selected_indices.append(chosen)
        previous_index = chosen
    selected_indices[0] = 0
    selected_indices[-1] = dense_count - 1
    if len(set(selected_indices)) != target_frame_count:
        raise RuntimeError("SSIM resampling produced duplicate selections")

    selected_directory = RIFE_WORK_DIRECTORY / "ssim_resampled_frames"
    selected_directory.mkdir(parents=True, exist_ok=True)
    # This is a generated, run-local staging directory. Clear numbered
    # frames so rerunning this cell cannot leave stale PNGs behind when
    # timing settings or the selected frame count change.
    for stale_frame in selected_directory.glob("*.png"):
        stale_frame.unlink()
    for output_index, dense_index in enumerate(selected_indices):
        source = RIFE_DENSE_PATHS[dense_index]
        destination = selected_directory / f"{output_index:07d}.png"
        try:
            os.link(source, destination)
        except OSError:
            shutil.copy2(source, destination)

    RIFE_FINAL_VIDEO_PATH = RIFE_RESULTS_DIRECTORY / "periodic_bspline_flowmorph_rife_ssim_loop.mp4"
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    subprocess.check_call([
        ffmpeg, "-y", "-framerate", str(RIFE_FINAL_FPS),
        "-i", str(selected_directory / "%07d.png"),
        "-an", "-c:v", "libx264", "-preset", "slow",
        "-crf", str(RIFE_VIDEO_CRF), "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", str(RIFE_FINAL_VIDEO_PATH),
    ])
    if not RIFE_FINAL_VIDEO_PATH.is_file() or RIFE_FINAL_VIDEO_PATH.stat().st_size == 0:
        raise RuntimeError("FFmpeg did not create the final video")

    selected_ssim = np.asarray([
        structural_similarity(
            dense_luma[selected_indices[index - 1]],
            dense_luma[selected_indices[index]],
            data_range=255,
        )
        for index in range(target_frame_count)
    ], dtype=np.float64)
    selected_motion = 1.0 - selected_ssim
    report = {
        "method": "cyclic PNGs -> Practical-RIFE -> circular 1-SSIM equal-motion sampling -> H.264",
        "rife_repository": RIFE_REPOSITORY_URL,
        "rife_revision": RIFE_REPOSITORY_REVISION,
        "rife_model_repository": RIFE_MODEL_REPOSITORY,
        "rife_model_revision": RIFE_MODEL_REVISION,
        "rife_model_filename": RIFE_MODEL_FILENAME,
        "rife_multiplier": RIFE_MULTIPLIER,
        "rife_batch_size": RIFE_BATCH_SIZE,
        "rife_scale": RIFE_SCALE,
        "rife_fp16": RIFE_USE_FP16,
        "base_frames": len(EXPORT_FRAME_PATHS),
        "dense_unique_frames": len(RIFE_DENSE_PATHS),
        "final_unique_frames": target_frame_count,
        "source_fps": SOURCE_SEQUENCE_FPS,
        "final_fps": RIFE_FINAL_FPS,
        "duration_seconds": target_frame_count / RIFE_FINAL_FPS,
        "dense_ssim": {
            "mean": float(circular_ssim.mean()),
            "median": float(np.median(circular_ssim)),
            "minimum": float(circular_ssim.min()),
            "wraparound": float(circular_ssim[0]),
        },
        "resampled_ssim": {
            "mean": float(selected_ssim.mean()),
            "median": float(np.median(selected_ssim)),
            "minimum": float(selected_ssim.min()),
            "wraparound": float(selected_ssim[0]),
            "motion_coefficient_of_variation": (
                float(selected_motion.std() / selected_motion.mean())
                if selected_motion.mean() else 0.0
            ),
        },
        "selected_dense_indices": selected_indices,
        "terminal_duplicate_in_video": False,
        "video": str(RIFE_FINAL_VIDEO_PATH),
    }
    RIFE_REPORT_PATH = RIFE_RESULTS_DIRECTORY / "rife_ssim_report.json"
    RIFE_REPORT_PATH.write_text(
        json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    RIFE_SSIM_PLOT_PATH = RIFE_RESULTS_DIRECTORY / "ssim_motion_profile.png"
    figure, axes = plt.subplots(2, 1, figsize=(12, 5), constrained_layout=True)
    axes[0].plot(motion_weights, linewidth=0.7, color="#4b6a88")
    axes[0].scatter([0], [motion_weights[0]], color="#c43d32", s=24, label="wrap edge")
    axes[0].set_title("Dense RIFE motion profile (1 − circular SSIM)")
    axes[0].legend(loc="upper right")
    axes[1].plot(selected_motion, linewidth=0.8, color="#7a5535")
    axes[1].scatter([0], [selected_motion[0]], color="#c43d32", s=24, label="wrap edge")
    axes[1].set_title("After equal-motion resampling")
    axes[1].set_xlabel("Frame edge")
    axes[1].legend(loc="upper right")
    for axis in axes:
        axis.set_ylabel("1 − SSIM")
        axis.grid(alpha=0.2)
    figure.savefig(RIFE_SSIM_PLOT_PATH, dpi=160, facecolor="white")
    plt.close(figure)

    plot_preview = Image.open(RIFE_SSIM_PLOT_PATH).convert("RGB")
    plot_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### Final RIFE + circular SSIM diagnostics"))
    display(plot_preview)
    del plot_preview
    display(Video(
        str(RIFE_FINAL_VIDEO_PATH),
        embed=False,
        width=RIFE_DISPLAY_WIDTH,
        html_attributes="controls loop muted playsinline",
    ))
    print({
        "final_video": str(RIFE_FINAL_VIDEO_PATH),
        "frames": target_frame_count,
        "fps": RIFE_FINAL_FPS,
        "seconds": round(target_frame_count / RIFE_FINAL_FPS, 3),
        "wraparound_ssim": round(float(selected_ssim[0]), 6),
        "motion_variation": round(report["resampled_ssim"]["motion_coefficient_of_variation"], 6),
    })

    if DOWNLOAD_FINAL_VIDEO:
        try:
            from google.colab import files
        except ImportError:
            print("Colab download helper unavailable; use the printed video path.")
        else:
            files.download(str(RIFE_FINAL_VIDEO_PATH))

    if not RIFE_KEEP_WORK_FRAMES:
        expected_parent = (Path(LOCAL_ASSET_ROOT) / PROJECT_NAME / "rife_work").resolve()
        target = RIFE_WORK_DIRECTORY.resolve()
        if target.parent != expected_parent:
            raise RuntimeError(f"Refusing to remove unexpected RIFE directory: {target}")
        shutil.rmtree(target)
        print("Removed temporary local RIFE PNGs after successful persistent export.")


## 15. Read-only circular flicker diagnosis

This analyzes the raw periodic-spline frames without modifying them and
saves the full plot and JSON report to Drive.


In [ ]:
from flowmorph_klein.flicker_diagnostics import (
    FlickerDiagnosticConfig,
    diagnose_cyclic_flicker,
    format_flicker_diagnostic_markdown,
)
from IPython.display import Markdown, display

if RUN_FLICKER_DIAGNOSTIC:
    diagnostic_records = globals().get("FINAL_RECORDS")
    if diagnostic_records is None:
        manifest_candidates = [
            RUN_DIRECTORY
            / "metadata"
            / "final_periodic_bspline_flowmorph_sequence.json",
            RUN_DIRECTORY
            / "metadata"
            / "final_periodic_bspline_flowmorph_sequence_tone_stabilized.json",
            RUN_DIRECTORY
            / "metadata"
            / "final_recursive_sequence.json",
        ]
        diagnostic_manifest = next(
            (path for path in manifest_candidates if path.is_file()),
            None,
        )
        if diagnostic_manifest is None:
            raise RuntimeError(
                "No final sequence is available for flicker diagnosis. "
                "Run the FlowMorph sequence or set RESUME_RUN_DIRECTORY "
                "to a completed run first."
            )
        diagnostic_records = json.loads(
            diagnostic_manifest.read_text(encoding="utf-8")
        )["records"]
        print("Restored diagnostic records from", diagnostic_manifest)

    final_gap_size = max(
        2,
        int(round(len(diagnostic_records) / BASE_PROMPT_COUNT)),
    )
    FLICKER_DIAGNOSTIC_RESULT = diagnose_cyclic_flicker(
        diagnostic_records,
        RUN_DIRECTORY / "diagnostics" / "flicker",
        config=FlickerDiagnosticConfig(
            analysis_max_side=FLICKER_ANALYSIS_MAX_SIDE,
            outlier_mad_multiplier=(
                FLICKER_OUTLIER_MAD_MULTIPLIER
            ),
            minimum_outlier_score=(
                FLICKER_MINIMUM_OUTLIER_SCORE
            ),
            max_lag=FLICKER_MAX_LAG,
            gap_size=final_gap_size,
            render_batch_size=FLOWMORPH_RENDER_BATCH_SIZE,
        ),
    )
    flicker_preview = Image.open(
        FLICKER_DIAGNOSTIC_RESULT.plot_path
    ).convert("RGB")
    flicker_preview.thumbnail(
        (CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000)
    )
    display(Markdown("### Raw FlowMorph flicker pattern diagnosis"))
    display(flicker_preview)
    flicker_preview.close()
    display(Markdown(format_flicker_diagnostic_markdown(
        FLICKER_DIAGNOSTIC_RESULT.report,
        max_pulse_centers=40,
        ranked_limit=10,
    )))
    strong_hypotheses = [
        item
        for item in FLICKER_DIAGNOSTIC_RESULT.report["hypotheses"]
        if item.get("support") == "strong"
    ]
    print({
        "raw_frames_analyzed": (
            FLICKER_DIAGNOSTIC_RESULT.report["frame_count"]
        ),
        "pulse_centers": (
            FLICKER_DIAGNOSTIC_RESULT.report["outlier_indices"]
        ),
        "strong_hypotheses": strong_hypotheses,
        "dominant_periods": (
            FLICKER_DIAGNOSTIC_RESULT.report["dominant_periods"][:5]
        ),
        "report": str(FLICKER_DIAGNOSTIC_RESULT.report_path),
        "plot": str(FLICKER_DIAGNOSTIC_RESULT.plot_path),
        "images_modified": False,
        "project_package_reload_required": False,
    })
else:
    print("Flicker diagnosis disabled.")
